# 2,000+ Score Baseline: The First Step on the Silver-Medal Path

> 🥈 **As of now, this approach is in the 2,000+ score / silver-medal range on the Kaggriculture leaderboard.**
> This notebook shares the baseline and the practical ideas behind that result: a complete farming route, market timing, resource management, and a few simple safeguards that make the plan reliable across episodes.


## What this notebook offers

The goal is to give you a solid starting point for reaching the 2,000+ and silver-medal range, while keeping the strategy easy to inspect and modify. The baseline demonstrates: (1) a complete 720-turn farming plan, (2) market orders as part of the plan, (3) defensive wheat/feed handling, (4) repairing weeds without losing the schedule, and (5) final liquidity management.

Run the final code cell to define `agent`, the callable expected by the Kaggle environment.


## Baseline design

The controller is open-loop at the strategic level: it follows a tested season route. Lightweight state-aware overlays handle execution details such as a weed appearing under a scheduled action, premium sell ordering, feed denial, banking, and terminal liquidation. Each overlay is visible in `CONFIG` and can be ablated.

This is the model I recommend using as a starting point for experiments—not the final leaderboard submission.


In [ ]:
# kaggriculture agent v1 — canonical-route tape + proven overlays.
# Self-contained, stdlib only. Built by build.py which injects TAPE and CONFIG.
#
# Overlays (each independently toggleable, all default on):
#   feed5        step-0 BUY_PRODUCT WHEAT moved to slot 0 (feed-denial defense)
#   weed_repair  scheduled tile op landing on a WEED -> DIG now, replay the op
#                next turn; lag absorbed at the actor's next scheduled PASS,
#                force-reset at day boundaries
#   sell_order   premium sells (MELON/STRAWBERRY/MILK/WOOL) promoted to the
#                front of the market list, sorted by self price impact;
#                WHEAT/FERTILIZER never promoted (opponent BUY_PRODUCT lifts them)
#   split        pull forward up to WHEAT 10 / FERTILIZER 5 from the next
#                turn's scheduled sells, with an exact debt ledger
#   banking      steps 120-679: an actor doing PASS/movement while carrying
#                >= $2000 of premium produce within 1 move of a shed tile
#                diverts to bank it
#   terminal     steps 714-718: walk carriers in, DROP, and sell everything
#                (step 718 is the last executed action; 719 never runs)
#   liq_ramp     from liq_start, add sells for shed stock the tape did not
#                schedule, spread over the remaining turns

import math

TAPE = [{'farmer': ['PASS'], 'hands': [], 'market': [['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_ANIMAL', 'COW', 1], ['BUY_ANIMAL', 'SHEEP', 4], ['BUY_SEED', 'WHEAT', 5], ['BUY_SEED', 'MELON', 5], ['BUY_PRODUCT', 'WHEAT', 5]]}, {'farmer': ['PICKUP', 'COW', 1], 'hands': [['WEST'], ['WEST'], ['PASS'], ['PICKUP', 'SHEEP', 4]], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 1], 'hands': [['NORTH'], ['NORTH'], ['PASS'], ['PICKUP', 'WHEAT', 4]], 'market': []}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['NORTH'], ['PASS'], ['BUILD_PASTURE']], 'market': []}, {'farmer': ['BUILD_PASTURE'], 'hands': [['NORTH'], ['NORTH'], ['PASS'], ['PLACE', 'SHEEP']], 'market': []}, {'farmer': ['PLACE', 'COW'], 'hands': [['PLANT', 'MELON'], ['NORTH'], ['PASS'], ['FEED', 'WHEAT']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['WATER'], ['NORTH'], ['PASS'], ['CARE']], 'market': []}, {'farmer': ['CARE'], 'hands': [['NORTH'], ['PLANT', 'WHEAT'], ['PASS'], ['NORTH']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PLANT', 'WHEAT'], ['WATER'], ['PASS'], ['BUILD_PASTURE']], 'market': []}, {'farmer': ['PASS'], 'hands': [['WATER'], ['WEST'], ['PASS'], ['PLACE', 'SHEEP']], 'market': []}, {'farmer': ['PASS'], 'hands': [['WEST'], ['PLANT', 'WHEAT'], ['PASS'], ['FEED', 'WHEAT']], 'market': []}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['WATER'], ['PASS'], ['CARE']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PLANT', 'MELON'], ['WEST'], ['PASS'], ['NORTH']], 'market': []}, {'farmer': ['PASS'], 'hands': [['WATER'], ['PLANT', 'WHEAT'], ['PASS'], ['BUILD_PASTURE']], 'market': []}, {'farmer': ['WEST'], 'hands': [['WEST'], ['WATER'], ['PASS'], ['PLACE', 'SHEEP']], 'market': []}, {'farmer': ['EAST'], 'hands': [['PLANT', 'MELON'], ['WEST'], ['PASS'], ['FEED', 'WHEAT']], 'market': []}, {'farmer': ['WEST'], 'hands': [['WATER'], ['PLANT', 'WHEAT'], ['PASS'], ['CARE']], 'market': []}, {'farmer': ['EAST'], 'hands': [['PASS'], ['WATER'], ['PASS'], ['WEST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['PASS'], ['SOUTH'], ['PASS'], ['SOUTH']], 'market': []}, {'farmer': ['EAST'], 'hands': [['PASS'], ['PLANT', 'MELON'], ['PASS'], ['BUILD_PASTURE']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['WATER'], ['PASS'], ['PLACE', 'SHEEP']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['EAST'], ['PASS'], ['FEED', 'WHEAT']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PLANT', 'MELON'], ['PASS'], ['CARE']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['WATER'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [], 'market': [['HIRE']]}, {'farmer': ['CARE'], 'hands': [['WEST']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['NORTH']], 'market': []}, {'farmer': ['WEST'], 'hands': [['CARE']], 'market': []}, {'farmer': ['CARE'], 'hands': [['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['NORTH']], 'market': []}, {'farmer': ['CARE'], 'hands': [['CARE']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['WEST']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['SOUTH']], 'market': []}, {'farmer': ['PASS'], 'hands': [['SOUTH']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'FERTILIZER', 5], ['HIRE'], ['HIRE'], ['BUY_PRODUCT', 'WHEAT', 6]]}, {'farmer': ['PICKUP', 'WHEAT', 1], 'hands': [['WEST'], ['NORTH']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['PICKUP', 'WHEAT', 4], ['NORTH']], 'market': []}, {'farmer': ['CARE'], 'hands': [['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['FEED', 'WHEAT'], ['NORTH']], 'market': []}, {'farmer': ['WEST'], 'hands': [['CARE'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['NORTH'], 'hands': [['COLLECT_FERTILIZER'], ['NORTH']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['NORTH'], ['WATER']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['FEED', 'WHEAT'], ['WEST']], 'market': []}, {'farmer': ['WATER'], 'hands': [['CARE'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['COLLECT_FERTILIZER'], ['WEST']], 'market': []}, {'farmer': ['WATER'], 'hands': [['WEST'], ['WATER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['SOUTH'], ['WEST']], 'market': []}, {'farmer': ['WATER'], 'hands': [['FEED', 'WHEAT'], ['WATER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['CARE'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WATER'], 'hands': [['COLLECT_FERTILIZER'], ['WATER']], 'market': []}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['FEED', 'WHEAT'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['CARE'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['COLLECT_FERTILIZER'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'FERTILIZER', 5], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_SEED', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 3]]}, {'farmer': ['PICKUP', 'WHEAT', 1], 'hands': [['WEST'], ['WEST'], ['WEST']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['PICKUP', 'WHEAT', 4], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['CARE'], 'hands': [['NORTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['FEED', 'WHEAT'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['WEST'], 'hands': [['CARE'], ['PLANT', 'STRAWBERRY'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['CARE'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['NORTH']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['NORTH'], ['WEST'], ['WATER']], 'market': []}, {'farmer': ['CARE'], 'hands': [['FEED', 'WHEAT'], ['PLANT', 'STRAWBERRY'], ['WEST']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['CARE'], ['WATER'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['WEST']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['WEST'], ['PLANT', 'STRAWBERRY'], ['WEST']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['SOUTH'], ['WATER'], ['SOUTH']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['FEED', 'WHEAT'], ['NORTH'], ['SOUTH']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['SOUTH'], ['NORTH'], ['PLANT', 'WHEAT']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['NORTH'], 'hands': [['FEED', 'WHEAT'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['WATER'], 'hands': [['PASS'], ['WEST'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['PASS'], ['WATER'], ['PASS']], 'market': []}, {'farmer': ['WATER'], 'hands': [['PASS'], ['EAST'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'FERTILIZER', 5], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 1], 'hands': [['WEST'], ['NORTH'], ['WEST']], 'market': [['BUY_SEED', 'WHEAT', 5]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['PICKUP', 'WHEAT', 4], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['CARE'], 'hands': [['NORTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['FEED', 'WHEAT'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['WEST'], 'hands': [['CARE'], ['WATER'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['NORTH']], 'market': []}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['FEED', 'WHEAT'], ['WEST'], ['HARVEST']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['CARE'], ['WATER'], ['PLANT', 'WHEAT']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['WATER']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['WEST'], ['WATER'], ['WEST']], 'market': []}, {'farmer': ['WATER'], 'hands': [['SOUTH'], ['WEST'], ['WATER']], 'market': []}, {'farmer': ['EAST'], 'hands': [['FEED', 'WHEAT'], ['NORTH'], ['HARVEST']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['CARE'], ['WATER'], ['PLANT', 'WHEAT']], 'market': []}, {'farmer': ['WATER'], 'hands': [['COLLECT_FERTILIZER'], ['HARVEST'], ['WATER']], 'market': []}, {'farmer': ['HARVEST'], 'hands': [['SOUTH'], ['PLANT', 'WHEAT'], ['WEST']], 'market': []}, {'farmer': ['PASS'], 'hands': [['FEED', 'WHEAT'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['PASS'], 'hands': [['CARE'], ['EAST'], ['HARVEST']], 'market': []}, {'farmer': ['PASS'], 'hands': [['COLLECT_FERTILIZER'], ['PLANT', 'WHEAT'], ['PLANT', 'WHEAT']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'WHEAT', 17], ['SELL', 'FERTILIZER', 5], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_ANIMAL', 'COW', 1], ['BUY_SEED', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 4]]}, {'farmer': ['PICKUP', 'COW', 1], 'hands': [['WEST'], ['WEST'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 4], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['NORTH'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['FEED', 'WHEAT'], ['NORTH'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['CARE'], ['WATER'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['PLANT', 'STRAWBERRY']], 'market': []}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['BUILD_PASTURE'], 'hands': [['FEED', 'WHEAT'], ['WEST'], ['WEST']], 'market': []}, {'farmer': ['PLACE', 'COW'], 'hands': [['CARE'], ['WATER'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['PLANT', 'WHEAT']], 'market': []}, {'farmer': ['CARE'], 'hands': [['WEST'], ['WATER'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['EAST'], ['EAST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['FEED', 'WHEAT'], ['SOUTH'], ['EAST']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['CARE'], ['PLANT', 'STRAWBERRY'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['EAST'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['FEED', 'WHEAT'], ['PLANT', 'STRAWBERRY'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['WATER'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['WEST'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['WEST'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['SOUTH'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PLANT', 'STRAWBERRY'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['WATER'], ['PASS']], 'market': []}, {'farmer': ['HARVEST'], 'hands': [], 'market': [['SELL', 'FERTILIZER', 5], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['DROP'], 'hands': [['WEST'], ['NORTH'], ['WEST']], 'market': [['SELL', 'WOOL', 5]]}, {'farmer': ['PICKUP', 'WHEAT', 3], 'hands': [['PICKUP', 'WHEAT', 3], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['NORTH'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['HARVEST'], ['NORTH'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['FEED', 'WHEAT'], ['WATER'], ['WEST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['CARE'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['NORTH']], 'market': []}, {'farmer': ['CARE'], 'hands': [['NORTH'], ['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['HARVEST'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['FEED', 'WHEAT'], ['WEST'], ['NORTH']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['CARE'], ['WATER'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['CARE'], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['WEST'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['SOUTH'], ['EAST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['HARVEST'], ['WATER'], ['EAST']], 'market': []}, {'farmer': ['DROP'], 'hands': [['FEED', 'WHEAT'], ['EAST'], ['EAST']], 'market': [['SELL', 'FERTILIZER', 3], ['BUY_LAND']]}, {'farmer': ['EAST'], 'hands': [['CARE'], ['EAST'], ['EAST']], 'market': [['HIRE'], ['BUY_ANIMAL', 'COW', 2], ['BUY_SEED', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 2], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'COW', 2], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['EAST'], ['EAST']], 'market': [['SELL', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['EAST'], ['WEST'], ['EAST'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['BUILD_PASTURE'], 'hands': [['SOUTH'], ['WATER'], ['PLANT', 'STRAWBERRY'], ['BUILD_PASTURE']], 'market': []}, {'farmer': ['PLACE', 'COW'], 'hands': [['PLACE', 'WOOL', 15], ['PASS'], ['WATER'], ['EAST']], 'market': [['SELL', 'WOOL', 15]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['PASS'], ['PASS'], ['EAST'], ['EAST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['PASS'], ['PASS'], ['PLANT', 'STRAWBERRY'], ['NORTH']], 'market': []}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'FERTILIZER', 3], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_ANIMAL', 'COW', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'COW', 3], ['WEST'], ['EAST'], ['PICKUP', 'WHEAT', 4], ['NORTH'], ['NORTH'], ['WEST']], 'market': [['BUY_SEED', 'WHEAT', 3], ['BUY_SEED', 'STRAWBERRY', 10], ['BUY_PRODUCT', 'WHEAT', 4]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['PICKUP', 'WHEAT', 4], ['NORTH'], ['NORTH'], ['NORTH'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['CARE'], 'hands': [['FEED', 'WHEAT'], ['NORTH'], ['NORTH'], ['FEED', 'WHEAT'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['CARE'], ['NORTH'], ['NORTH'], ['CARE'], ['PLANT', 'STRAWBERRY'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['WEST'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['NORTH'], ['COLLECT_FERTILIZER'], ['WATER'], ['WATER'], ['NORTH']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['NORTH'], ['WEST'], ['PLANT', 'STRAWBERRY'], ['NORTH'], ['EAST'], ['WEST'], ['NORTH']], 'market': []}, {'farmer': ['CARE'], 'hands': [['BUILD_PASTURE'], ['WATER'], ['WATER'], ['FEED', 'WHEAT'], ['SOUTH'], ['WATER'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['PLACE', 'COW'], ['WEST'], ['EAST'], ['CARE'], ['PLANT', 'STRAWBERRY'], ['WEST'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['FEED', 'WHEAT'], ['WATER'], ['PLANT', 'STRAWBERRY'], ['COLLECT_FERTILIZER'], ['WATER'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['CARE'], ['SOUTH'], ['WATER'], ['WEST'], ['EAST'], ['WEST'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WATER'], 'hands': [['NORTH'], ['WATER'], ['EAST'], ['SOUTH'], ['PLANT', 'STRAWBERRY'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['BUILD_PASTURE'], ['WEST'], ['DIG'], ['FEED', 'WHEAT'], ['WATER'], ['WEST'], ['EAST']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['PLACE', 'COW'], ['WATER'], ['PLANT', 'STRAWBERRY'], ['CARE'], ['EAST'], ['WATER'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WATER'], 'hands': [['FEED', 'WHEAT'], ['SOUTH'], ['WATER'], ['COLLECT_FERTILIZER'], ['PLANT', 'STRAWBERRY'], ['SOUTH'], ['EAST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['CARE'], ['WATER'], ['EAST'], ['WEST'], ['WATER'], ['WATER'], ['PLANT', 'WHEAT']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['CARE'], 'hands': [['EAST'], ['EAST'], ['PLANT', 'STRAWBERRY'], ['SOUTH'], ['WEST'], ['HARVEST'], ['WATER']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['SOUTH'], ['NORTH'], ['WATER'], ['FEED', 'WHEAT'], ['NORTH'], ['PLANT', 'WHEAT'], ['EAST']], 'market': []}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['NORTH'], ['NORTH'], ['PASS'], ['NORTH'], ['WATER'], ['EAST']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PLACE', 'COW'], ['NORTH'], ['PLANT', 'STRAWBERRY'], ['PASS'], ['DIG'], ['NORTH'], ['EAST']], 'market': []}, {'farmer': ['PASS'], 'hands': [['FEED', 'WHEAT'], ['NORTH'], ['WATER'], ['PASS'], ['PLANT', 'WHEAT'], ['NORTH'], ['PLANT', 'WHEAT']], 'market': []}, {'farmer': ['PASS'], 'hands': [['CARE'], ['WATER'], ['SOUTH'], ['PASS'], ['WATER'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PLANT', 'STRAWBERRY'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'FERTILIZER', 7], ['SELL', 'WHEAT', 2], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_ANIMAL', 'COW', 2], ['BUY_SEED', 'WHEAT', 3]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 4], ['NORTH'], ['EAST'], ['PICKUP', 'WHEAT', 4], ['PICKUP', 'COW', 2], ['NORTH']], 'market': [['BUY_SEED', 'WHEAT', 5], ['BUY_SEED', 'STRAWBERRY', 2], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['FEED', 'WHEAT'], ['NORTH'], ['NORTH'], ['WEST'], ['PICKUP', 'WHEAT', 2], ['NORTH']], 'market': []}, {'farmer': ['CARE'], 'hands': [['CARE'], ['NORTH'], ['NORTH'], ['HARVEST'], ['EAST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['FEED', 'WHEAT'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['NORTH'], ['WATER'], ['NORTH'], ['CARE'], ['BUILD_PASTURE'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['FEED', 'WHEAT'], ['WEST'], ['NORTH'], ['COLLECT_FERTILIZER'], ['PLACE', 'COW'], ['WATER']], 'market': []}, {'farmer': ['CARE'], 'hands': [['CARE'], ['WATER'], ['WATER'], ['NORTH'], ['FEED', 'WHEAT'], ['HARVEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['EAST'], ['FEED', 'WHEAT'], ['CARE'], ['PLANT', 'WHEAT']], 'market': []}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['WATER'], ['EAST'], ['CARE'], ['EAST'], ['WATER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['FEED', 'WHEAT'], ['WEST'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['WEST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['CARE'], ['WATER'], ['SOUTH'], ['EAST'], ['BUILD_PASTURE'], ['WATER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['SOUTH'], ['NORTH'], ['PLACE', 'COW'], ['HARVEST']], 'market': []}, {'farmer': ['WATER'], 'hands': [['EAST'], ['WATER'], ['SOUTH'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['PLANT', 'WHEAT']], 'market': []}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['NORTH'], ['PLANT', 'STRAWBERRY'], ['CARE'], ['CARE'], ['WATER']], 'market': []}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['WATER'], ['WATER'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['FEED', 'WHEAT'], ['HARVEST'], ['EAST'], ['WEST'], ['PLANT', 'WHEAT'], ['WATER']], 'market': []}, {'farmer': ['CARE'], 'hands': [['CARE'], ['PLANT', 'WHEAT'], ['PLANT', 'STRAWBERRY'], ['WEST'], ['WATER'], ['HARVEST']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['WATER'], ['SOUTH'], ['EAST'], ['PLANT', 'WHEAT']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['EAST'], ['NORTH'], ['SOUTH'], ['PLANT', 'WHEAT'], ['WATER']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['WATER'], ['PLANT', 'WHEAT'], ['FEED', 'WHEAT'], ['WATER'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['HARVEST'], ['WATER'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PLANT', 'WHEAT'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['WATER'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'MILK', 6], ['SELL', 'FERTILIZER', 10], ['SELL', 'WHEAT', 13], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['PICKUP', 'WHEAT', 4], ['PICKUP', 'WHEAT', 4], ['NORTH'], ['EAST']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['FEED', 'WHEAT'], ['NORTH'], ['NORTH'], ['HARVEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['CARE'], ['NORTH'], ['NORTH'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['CARE'], 'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['CARE'], ['CARE'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['EAST'], ['WATER'], ['WATER'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['WATER'], ['NORTH']], 'market': []}, {'farmer': ['WEST'], 'hands': [['FEED', 'WHEAT'], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH'], ['WEST'], ['WATER']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['CARE'], ['WATER'], ['WATER'], ['HARVEST'], ['FEED', 'WHEAT'], ['WATER'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['CARE'], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['EAST'], ['FEED', 'WHEAT'], ['CARE'], ['WEST'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['EAST'], ['WATER'], ['SOUTH'], ['CARE'], ['COLLECT_FERTILIZER'], ['WATER'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['NORTH'], 'hands': [['NORTH'], ['WEST'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['EAST'], ['WEST'], ['WATER']], 'market': []}, {'farmer': ['WATER'], 'hands': [['NORTH'], ['WATER'], ['WATER'], ['NORTH'], ['SOUTH'], ['WATER'], ['EAST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['WATER'], ['EAST'], ['EAST'], ['HARVEST'], ['FEED', 'WHEAT'], ['WEST'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WATER'], 'hands': [['EAST'], ['SOUTH'], ['EAST'], ['FEED', 'WHEAT'], ['CARE'], ['WATER'], ['EAST']], 'market': [['SELL', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['WATER'], ['SOUTH'], ['NORTH'], ['CARE'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['SOUTH'], 'hands': [['NORTH'], ['WATER'], ['EAST'], ['COLLECT_FERTILIZER'], ['EAST'], ['SOUTH'], ['SOUTH']], 'market': []}, {'farmer': ['WATER'], 'hands': [['WATER'], ['WEST'], ['SOUTH'], ['WEST'], ['SOUTH'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['DIG'], ['SOUTH'], ['FEED', 'WHEAT'], ['HARVEST'], ['SOUTH']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['HARVEST'], ['CARE'], ['PLANT', 'WHEAT'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['FEED', 'WHEAT'], ['COLLECT_FERTILIZER'], ['WATER'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['CARE'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['EAST'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PLACE', 'WOOL', 16], ['PASS'], ['PASS'], ['PASS']], 'market': [['SELL', 'WOOL', 16]]}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'WOOL', 16], ['SELL', 'WHEAT', 2], ['BUY_LAND'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['SELL', 'FERTILIZER', 9], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_SEED', 'MELON', 9], ['BUY_SEED', 'STRAWBERRY', 7]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['SOUTH'], ['WEST'], ['PICKUP', 'WHEAT', 4], ['PICKUP', 'WHEAT', 4], ['WEST'], ['WEST'], ['NORTH'], ['EAST'], ['WEST'], ['WEST'], ['NORTH'], ['NORTH'], ['WEST']], 'market': [['BUY_SEED', 'MELON', 1]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['FEED', 'WHEAT'], ['PLANT', 'STRAWBERRY'], ['WEST'], ['NORTH'], ['NORTH'], ['WEST'], ['WEST'], ['NORTH'], ['EAST'], ['NORTH'], ['WEST'], ['NORTH'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['CARE'], ['WATER'], ['WEST'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['SOUTH'], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH'], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['COLLECT_FERTILIZER'], ['SOUTH'], ['WEST'], ['CARE'], ['CARE'], ['SOUTH'], ['WEST'], ['WATER'], ['WATER'], ['NORTH'], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['WEST'], 'hands': [['EAST'], ['PLANT', 'STRAWBERRY'], ['WEST'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['PLANT', 'STRAWBERRY'], ['NORTH'], ['HARVEST'], ['EAST'], ['NORTH'], ['WEST'], ['WATER'], ['WATER'], ['NORTH']], 'market': []}, {'farmer': ['WEST'], 'hands': [['FEED', 'WHEAT'], ['WATER'], ['SOUTH'], ['NORTH'], ['NORTH'], ['WATER'], ['NORTH'], ['PLANT', 'MELON'], ['WATER'], ['WATER'], ['NORTH'], ['WEST'], ['EAST'], ['NORTH']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['CARE'], ['WEST'], ['SOUTH'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['WEST'], ['NORTH'], ['WATER'], ['EAST'], ['HARVEST'], ['NORTH'], ['WATER'], ['EAST'], ['WATER']], 'market': []}, {'farmer': ['CARE'], 'hands': [['COLLECT_FERTILIZER'], ['PLANT', 'STRAWBERRY'], ['PLANT', 'STRAWBERRY'], ['CARE'], ['CARE'], ['PLANT', 'STRAWBERRY'], ['NORTH'], ['SOUTH'], ['WATER'], ['PLANT', 'MELON'], ['NORTH'], ['WEST'], ['WATER'], ['HARVEST']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['WEST'], ['WATER'], ['WATER'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['WATER'], ['WATER'], ['SOUTH'], ['SOUTH'], ['WATER'], ['NORTH'], ['WATER'], ['EAST'], ['PLANT', 'MELON']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['WEST'], ['SOUTH'], ['SOUTH'], ['WEST'], ['EAST'], ['EAST'], ['HARVEST'], ['SOUTH'], ['WATER'], ['EAST'], ['WATER'], ['WEST'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['PLANT', 'MELON'], 'hands': [['SOUTH'], ['PLANT', 'STRAWBERRY'], ['PASS'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['PLANT', 'MELON'], ['DROP'], ['WEST'], ['SOUTH'], ['HARVEST'], ['WATER'], ['WEST'], ['EAST']], 'market': [['SELL', 'MELON', 6], ['BUY_SEED', 'MELON', 3], ['BUY_SEED', 'STRAWBERRY', 9], ['BUY_PRODUCT', 'WHEAT', 4]]}, {'farmer': ['WATER'], 'hands': [['PLANT', 'MELON'], ['WATER'], ['PLANT', 'STRAWBERRY'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['PLANT', 'STRAWBERRY'], ['WATER'], ['EAST'], ['WATER'], ['SOUTH'], ['PLANT', 'MELON'], ['WEST'], ['SOUTH'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 4]]}, {'farmer': ['WEST'], 'hands': [['WATER'], ['EAST'], ['WATER'], ['CARE'], ['CARE'], ['WATER'], ['EAST'], ['EAST'], ['WEST'], ['SOUTH'], ['WATER'], ['WATER'], ['SOUTH'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PLANT', 'MELON'], 'hands': [['WEST'], ['PLANT', 'STRAWBERRY'], ['EAST'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['EAST'], ['NORTH'], ['WEST'], ['DROP'], ['EAST'], ['SOUTH'], ['SOUTH'], ['SOUTH']], 'market': [['SELL', 'MELON', 6]]}, {'farmer': ['WATER'], 'hands': [['PLANT', 'MELON'], ['WATER'], ['PLANT', 'STRAWBERRY'], ['SOUTH'], ['EAST'], ['PLANT', 'STRAWBERRY'], ['EAST'], ['NORTH'], ['NORTH'], ['WEST'], ['EAST'], ['SOUTH'], ['SOUTH'], ['SOUTH']], 'market': []}, {'farmer': ['WEST'], 'hands': [['WATER'], ['SOUTH'], ['WATER'], ['HARVEST'], ['SOUTH'], ['WATER'], ['SOUTH'], ['NORTH'], ['NORTH'], ['CARE'], ['EAST'], ['WATER'], ['CARE'], ['DROP']], 'market': [['SELL', 'MELON', 6]]}, {'farmer': ['PLANT', 'MELON'], 'hands': [['SOUTH'], ['PLANT', 'STRAWBERRY'], ['SOUTH'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['EAST'], ['SOUTH'], ['NORTH'], ['NORTH'], ['COLLECT_FERTILIZER'], ['EAST'], ['PASS'], ['COLLECT_FERTILIZER'], ['PASS']], 'market': []}, {'farmer': ['WATER'], 'hands': [['PLANT', 'MELON'], ['WATER'], ['PLANT', 'STRAWBERRY'], ['PASS'], ['PASS'], ['PLANT', 'STRAWBERRY'], ['SOUTH'], ['WATER'], ['NORTH'], ['PASS'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['SOUTH'], 'hands': [['WATER'], ['PASS'], ['WATER'], ['PASS'], ['PASS'], ['WATER'], ['DROP'], ['PASS'], ['PASS'], ['PASS'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS']], 'market': [['SELL', 'MELON', 6]]}, {'farmer': ['PLANT', 'MELON'], 'hands': [['PASS'], ['PASS'], ['WEST'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['WATER'], 'hands': [['PASS'], ['PASS'], ['PLANT', 'STRAWBERRY'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['DROP'], ['PASS'], ['PASS'], ['PASS']], 'market': [['SELL', 'MELON', 6], ['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['PASS'], ['PASS'], ['WATER'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'MILK', 3], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['WEST'], ['EAST'], ['PICKUP', 'WHEAT', 4], ['PICKUP', 'WHEAT', 4], ['WEST'], ['NORTH'], ['WEST'], ['NORTH'], ['WEST']], 'market': [['BUY_SEED', 'WHEAT', 3]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['FEED', 'WHEAT'], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH'], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['CARE'], ['SOUTH'], ['NORTH'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['COLLECT_FERTILIZER'], ['PLANT', 'MELON'], ['NORTH'], ['CARE'], ['CARE'], ['WEST'], ['NORTH'], ['WATER'], ['WATER'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['WEST'], 'hands': [['EAST'], ['WATER'], ['WATER'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['WEST'], ['EAST'], ['WATER']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['FEED', 'WHEAT'], ['WEST'], ['EAST'], ['NORTH'], ['NORTH'], ['NORTH'], ['WATER'], ['WATER'], ['WATER'], ['WEST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['CARE'], ['PASS'], ['NORTH'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['NORTH'], ['HARVEST'], ['WEST'], ['EAST'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['NORTH'], ['CARE'], ['CARE'], ['WATER'], ['PLANT', 'WHEAT'], ['WATER'], ['WATER'], ['WEST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['EAST'], ['WEST'], ['WATER'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['WATER'], ['WEST'], ['EAST'], ['WATER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['EAST'], ['HARVEST'], ['WEST'], ['EAST'], ['PLANT', 'WHEAT'], ['EAST'], ['SOUTH'], ['WATER'], ['SOUTH']], 'market': []}, {'farmer': ['WATER'], 'hands': [['NORTH'], ['EAST'], ['PLANT', 'WHEAT'], ['SOUTH'], ['SOUTH'], ['WATER'], ['EAST'], ['SOUTH'], ['EAST'], ['WATER']], 'market': []}, {'farmer': ['EAST'], 'hands': [['EAST'], ['NORTH'], ['WATER'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['NORTH'], ['SOUTH'], ['EAST'], ['WATER'], ['EAST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['WATER'], ['NORTH'], ['EAST'], ['CARE'], ['CARE'], ['NORTH'], ['SOUTH'], ['EAST'], ['NORTH'], ['EAST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['WEST'], ['COLLECT_FERTILIZER'], ['WATER'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['WATER'], ['SOUTH'], ['EAST'], ['WATER'], ['EAST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['WATER'], ['EAST'], ['HARVEST'], ['WEST'], ['EAST'], ['EAST'], ['WATER'], ['NORTH'], ['SOUTH'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['EAST'], ['PLANT', 'WHEAT'], ['SOUTH'], ['SOUTH'], ['WATER'], ['EAST'], ['NORTH'], ['SOUTH'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['NORTH'], ['WATER'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['EAST'], ['WATER'], ['NORTH'], ['WATER'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['EAST'], ['PASS'], ['CARE'], ['WATER'], ['EAST'], ['NORTH'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['NORTH'], ['SOUTH'], ['PASS'], ['PASS'], ['EAST'], ['WATER'], ['WATER'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['EAST'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['WATER'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'FERTILIZER', 12], ['SELL', 'WHEAT', 10], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['HIRE'], ['HIRE'], ['HIRE'], ['BUY_ANIMAL', 'COW', 1], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 3], 'hands': [['PICKUP', 'WHEAT', 2], ['WATER'], ['EAST'], ['PICKUP', 'WHEAT', 3], ['PICKUP', 'WHEAT', 4], ['PICKUP', 'FERTILIZER', 1], ['WEST'], ['PICKUP', 'FERTILIZER', 2], ['EAST'], ['WEST'], ['WEST']], 'market': [['BUY_SEED', 'WHEAT', 8]]}, {'farmer': ['HARVEST'], 'hands': [['FEED', 'WHEAT'], ['WEST'], ['EAST'], ['NORTH'], ['NORTH'], ['WEST'], ['WEST'], ['NORTH'], ['EAST'], ['WEST'], ['NORTH']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['CARE'], ['WATER'], ['NORTH'], ['HARVEST'], ['FEED', 'WHEAT'], ['NORTH'], ['WEST'], ['NORTH'], ['EAST'], ['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['CARE'], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['NORTH'], ['FEED', 'WHEAT'], ['CARE'], ['NORTH'], ['WEST'], ['NORTH'], ['WATER'], ['WATER'], ['NORTH']], 'market': [['BUY_SEED', 'STRAWBERRY', 1], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['EAST'], ['WATER'], ['WATER'], ['CARE'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['WATER'], ['EAST'], ['EAST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['FEED', 'WHEAT'], ['EAST'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['NORTH'], ['FERTILIZE', 'FERTILIZER'], ['WATER'], ['WEST'], ['WATER'], ['SOUTH'], ['NORTH']], 'market': []}, {'farmer': ['HARVEST'], 'hands': [['CARE'], ['SOUTH'], ['PLANT', 'WHEAT'], ['NORTH'], ['FEED', 'WHEAT'], ['WATER'], ['SOUTH'], ['WATER'], ['WEST'], ['SOUTH'], ['WATER']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['WATER'], ['HARVEST'], ['CARE'], ['WEST'], ['WATER'], ['WEST'], ['WEST'], ['WATER'], ['HARVEST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['WEST'], ['EAST'], ['EAST'], ['FEED', 'WHEAT'], ['COLLECT_FERTILIZER'], ['WEST'], ['SOUTH'], ['WATER'], ['WEST'], ['EAST'], ['PLANT', 'WHEAT']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['WEST'], ['WATER'], ['WATER'], ['CARE'], ['EAST'], ['NORTH'], ['WATER'], ['WEST'], ['NORTH'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['WEST'], ['SOUTH'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['NORTH'], ['EAST'], ['WATER'], ['NORTH'], ['SOUTH'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['WEST'], ['WATER'], ['PLANT', 'WHEAT'], ['WEST'], ['FEED', 'WHEAT'], ['WATER'], ['WATER'], ['WEST'], ['NORTH'], ['WATER'], ['WATER']], 'market': [['SELL', 'WHEAT', 1]]}, {'farmer': ['CARE'], 'hands': [['WEST'], ['SOUTH'], ['WATER'], ['SOUTH'], ['CARE'], ['HARVEST'], ['SOUTH'], ['WATER'], ['NORTH'], ['WEST'], ['HARVEST']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['WEST'], ['WATER'], ['EAST'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['PLANT', 'WHEAT'], ['WATER'], ['EAST'], ['WATER'], ['WATER'], ['PLANT', 'WHEAT']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['NORTH'], ['SOUTH'], ['WATER'], ['FEED', 'WHEAT'], ['EAST'], ['WATER'], ['WEST'], ['SOUTH'], ['PASS'], ['EAST'], ['WATER']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['WATER'], ['WATER'], ['HARVEST'], ['CARE'], ['SOUTH'], ['WEST'], ['WATER'], ['FERTILIZE', 'FERTILIZER'], ['PASS'], ['EAST'], ['WEST']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['PASS'], ['WEST'], ['PLANT', 'WHEAT'], ['COLLECT_FERTILIZER'], ['FEED', 'WHEAT'], ['WATER'], ['SOUTH'], ['WATER'], ['PASS'], ['WEST'], ['WATER']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['PASS'], ['WATER'], ['WATER'], ['PASS'], ['CARE'], ['HARVEST'], ['WATER'], ['EAST'], ['PASS'], ['SOUTH'], ['HARVEST']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['EAST'], ['COLLECT_FERTILIZER'], ['PLANT', 'WHEAT'], ['EAST'], ['FERTILIZE', 'FERTILIZER'], ['PASS'], ['PASS'], ['PLANT', 'WHEAT']], 'market': []}, {'farmer': ['WATER'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['SOUTH'], ['PASS'], ['WATER'], ['WATER'], ['WATER'], ['PASS'], ['PASS'], ['WATER']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['DROP'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['SELL', 'WOOL', 12]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'WOOL', 4], ['SELL', 'WHEAT', 25], ['SELL', 'FERTILIZER', 9], ['SELL', 'MILK', 3], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['HIRE'], ['HIRE'], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['PICKUP', 'WHEAT', 4], ['PICKUP', 'WHEAT', 4], ['WEST'], ['EAST'], ['WEST']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['FEED', 'WHEAT'], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH'], ['NORTH'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['CARE'], ['SOUTH'], ['NORTH'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['NORTH'], ['CARE'], ['CARE'], ['NORTH'], ['NORTH'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['WEST'], 'hands': [['EAST'], ['WEST'], ['WATER'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['FEED', 'WHEAT'], ['WATER'], ['NORTH'], ['NORTH'], ['NORTH'], ['WEST'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['CARE'], 'hands': [['CARE'], ['WEST'], ['WATER'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['HARVEST'], ['EAST'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['EAST'], ['CARE'], ['CARE'], ['EAST'], ['WATER'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['EAST'], 'hands': [['EAST'], ['NORTH'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['EAST'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['NORTH'], ['NORTH'], ['SOUTH'], ['WEST'], ['EAST'], ['EAST'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['EAST'], 'hands': [['NORTH'], ['NORTH'], ['WATER'], ['SOUTH'], ['SOUTH'], ['EAST'], ['EAST'], ['NORTH']], 'market': []}, {'farmer': ['EAST'], 'hands': [['WATER'], ['WATER'], ['EAST'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['EAST'], ['WATER'], ['NORTH']], 'market': []}, {'farmer': ['EAST'], 'hands': [['EAST'], ['EAST'], ['EAST'], ['CARE'], ['CARE'], ['SOUTH'], ['EAST'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['NORTH'], 'hands': [['NORTH'], ['EAST'], ['EAST'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['WATER'], ['HARVEST']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['WATER'], ['SOUTH'], ['WATER'], ['WEST'], ['EAST'], ['CARE'], ['SOUTH'], ['WEST']], 'market': []}, {'farmer': ['WATER'], 'hands': [['PASS'], ['SOUTH'], ['PASS'], ['SOUTH'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['WATER'], ['SOUTH']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['CARE'], ['PASS'], ['HARVEST'], ['FEED', 'WHEAT'], ['PASS'], ['PASS'], ['WATER']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['COLLECT_FERTILIZER'], ['PASS'], ['FEED', 'WHEAT'], ['PASS'], ['PASS'], ['PASS'], ['HARVEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PLANT', 'WHEAT']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['WATER']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'MILK', 6], ['SELL', 'FERTILIZER', 12], ['SELL', 'STRAWBERRY', 6], ['SELL', 'WHEAT', 4], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['WATER'], ['WEST'], ['PICKUP', 'WHEAT', 4], ['PICKUP', 'WHEAT', 4], ['PICKUP', 'FERTILIZER', 2], ['WEST'], ['PICKUP', 'FERTILIZER', 2], ['EAST'], ['WEST']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['NORTH'], ['WEST'], ['WEST'], ['NORTH'], ['HARVEST'], ['WEST'], ['WEST'], ['NORTH'], ['EAST'], ['WEST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['FEED', 'WHEAT'], ['WATER'], ['WEST'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['CARE'], ['WEST'], ['WEST'], ['CARE'], ['CARE'], ['WEST'], ['NORTH'], ['NORTH'], ['WATER'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['WEST'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['WEST'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['WATER'], ['EAST'], ['EAST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['EAST'], ['EAST'], ['WATER'], ['NORTH'], ['NORTH'], ['FERTILIZE', 'FERTILIZER'], ['NORTH'], ['NORTH'], ['WATER'], ['SOUTH']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['SOUTH'], ['SOUTH'], ['SOUTH'], ['FEED', 'WHEAT'], ['NORTH'], ['WATER'], ['NORTH'], ['WATER'], ['EAST'], ['SOUTH']], 'market': []}, {'farmer': ['CARE'], 'hands': [['FEED', 'WHEAT'], ['WATER'], ['WATER'], ['CARE'], ['FEED', 'WHEAT'], ['WEST'], ['WATER'], ['WEST'], ['WATER'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['CARE'], ['EAST'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['CARE'], ['FERTILIZE', 'FERTILIZER'], ['WEST'], ['SOUTH'], ['SOUTH'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['NORTH'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['WATER'], ['WEST'], ['COLLECT_FERTILIZER'], ['WATER'], ['WATER'], ['WATER'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['WEST'], ['SOUTH'], ['EAST'], ['SOUTH'], ['EAST'], ['NORTH'], ['WEST'], ['SOUTH'], ['WEST'], ['SOUTH']], 'market': []}, {'farmer': ['WATER'], 'hands': [['NORTH'], ['WATER'], ['WATER'], ['FEED', 'WHEAT'], ['SOUTH'], ['NORTH'], ['WATER'], ['WATER'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['NORTH'], ['SOUTH'], ['SOUTH'], ['CARE'], ['FEED', 'WHEAT'], ['WATER'], ['WEST'], ['WEST'], ['WEST'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WATER'], 'hands': [['NORTH'], ['WATER'], ['WATER'], ['COLLECT_FERTILIZER'], ['CARE'], ['NORTH'], ['WATER'], ['SOUTH'], ['WEST'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['SOUTH'], ['WEST'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['WATER'], ['SOUTH'], ['FERTILIZE', 'FERTILIZER'], ['NORTH'], ['EAST']], 'market': []}, {'farmer': ['WATER'], 'hands': [['WATER'], ['WATER'], ['WATER'], ['HARVEST'], ['EAST'], ['EAST'], ['EAST'], ['WATER'], ['NORTH'], ['EAST']], 'market': []}, {'farmer': ['PASS'], 'hands': [['EAST'], ['WEST'], ['SOUTH'], ['FEED', 'WHEAT'], ['SOUTH'], ['PASS'], ['SOUTH'], ['WEST'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['PASS'], 'hands': [['EAST'], ['WATER'], ['WATER'], ['CARE'], ['FEED', 'WHEAT'], ['PASS'], ['WATER'], ['FERTILIZE', 'FERTILIZER'], ['NORTH'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['WATER'], ['WEST'], ['EAST'], ['COLLECT_FERTILIZER'], ['CARE'], ['PASS'], ['PASS'], ['WATER'], ['WATER'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['EAST'], ['WATER'], ['WATER'], ['PASS'], ['COLLECT_FERTILIZER'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['SELL', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['WATER'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['SELL', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'MILK', 9], ['SELL', 'FERTILIZER', 8], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['HIRE'], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 3], 'hands': [['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['PICKUP', 'WHEAT', 3], ['PICKUP', 'WHEAT', 4], ['WEST'], ['NORTH'], ['WEST'], ['PICKUP', 'FERTILIZER', 1]], 'market': [['BUY_SEED', 'WHEAT', 4]]}, {'farmer': ['HARVEST'], 'hands': [['FEED', 'WHEAT'], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH'], ['WEST'], ['NORTH'], ['NORTH'], ['EAST']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['CARE'], ['SOUTH'], ['NORTH'], ['NORTH'], ['HARVEST'], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['CARE'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['NORTH'], ['HARVEST'], ['FEED', 'WHEAT'], ['WEST'], ['NORTH'], ['HARVEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['EAST'], ['WEST'], ['WATER'], ['FEED', 'WHEAT'], ['CARE'], ['NORTH'], ['NORTH'], ['WEST'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['NORTH'], 'hands': [['EAST'], ['WATER'], ['EAST'], ['CARE'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WATER'], ['SOUTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['HARVEST'], 'hands': [['FEED', 'WHEAT'], ['EAST'], ['WATER'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WATER'], ['HARVEST'], ['HARVEST'], ['NORTH']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['CARE'], ['EAST'], ['EAST'], ['WEST'], ['HARVEST'], ['NORTH'], ['PLANT', 'WHEAT'], ['WEST'], ['FERTILIZE', 'FERTILIZER']], 'market': []}, {'farmer': ['CARE'], 'hands': [['COLLECT_FERTILIZER'], ['EAST'], ['WATER'], ['SOUTH'], ['FEED', 'WHEAT'], ['WATER'], ['WATER'], ['SOUTH'], ['WATER']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['NORTH'], ['NORTH'], ['EAST'], ['HARVEST'], ['CARE'], ['HARVEST'], ['EAST'], ['HARVEST'], ['EAST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['NORTH'], ['WATER'], ['FEED', 'WHEAT'], ['COLLECT_FERTILIZER'], ['PLANT', 'WHEAT'], ['EAST'], ['NORTH'], ['WATER']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['WATER'], ['NORTH'], ['EAST'], ['CARE'], ['EAST'], ['WATER'], ['SOUTH'], ['NORTH'], ['HARVEST']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['EAST'], ['NORTH'], ['WATER'], ['COLLECT_FERTILIZER'], ['WEST'], ['EAST'], ['SOUTH'], ['NORTH'], ['PLANT', 'WHEAT']], 'market': []}, {'farmer': ['CARE'], 'hands': [['WATER'], ['NORTH'], ['NORTH'], ['WEST'], ['SOUTH'], ['HARVEST'], ['SOUTH'], ['NORTH'], ['WATER']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['EAST'], ['NORTH'], ['WATER'], ['SOUTH'], ['SOUTH'], ['EAST'], ['WATER'], ['WATER'], ['EAST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['WATER'], ['WATER'], ['SOUTH'], ['HARVEST'], ['DROP'], ['HARVEST'], ['EAST'], ['WEST'], ['WATER']], 'market': [['SELL', 'MILK', 12], ['SELL', 'FERTILIZER', 1]]}, {'farmer': ['CARE'], 'hands': [['WEST'], ['WEST'], ['PASS'], ['FEED', 'WHEAT'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['WATER'], ['WATER'], ['HARVEST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['WEST'], ['WATER'], ['PASS'], ['COLLECT_FERTILIZER'], ['EAST'], ['EAST'], ['EAST'], ['EAST'], ['PLANT', 'WHEAT']], 'market': []}, {'farmer': ['EAST'], 'hands': [['WEST'], ['PASS'], ['PASS'], ['WEST'], ['NORTH'], ['EAST'], ['WATER'], ['EAST'], ['WATER']], 'market': []}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['PASS'], ['PASS'], ['NORTH'], ['FEED', 'WHEAT'], ['EAST'], ['PASS'], ['WATER'], ['PASS']], 'market': []}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['PASS'], ['PASS'], ['HARVEST'], ['CARE'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['CARE'], 'hands': [['HARVEST'], ['PASS'], ['PASS'], ['PASS'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'WOOL', 16], ['SELL', 'STRAWBERRY', 12], ['SELL', 'MILK', 9], ['SELL', 'FERTILIZER', 9], ['SELL', 'WHEAT', 10], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_SEED', 'WHEAT', 3]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['WATER'], ['NORTH'], ['PICKUP', 'WHEAT', 4], ['PICKUP', 'WHEAT', 4], ['WEST'], ['EAST'], ['PICKUP', 'FERTILIZER', 3], ['PICKUP', 'FERTILIZER', 4], ['WEST'], ['WEST'], ['NORTH'], ['PICKUP', 'FERTILIZER', 5], ['NORTH']], 'market': [['BUY_SEED', 'WHEAT', 5], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['NORTH'], ['WEST'], ['PICKUP', 'FERTILIZER', 1], ['NORTH'], ['HARVEST'], ['WEST'], ['EAST'], ['WEST'], ['EAST'], ['WEST'], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['CARE'], 'hands': [['FEED', 'WHEAT'], ['WATER'], ['EAST'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['WEST'], ['NORTH'], ['NORTH'], ['EAST'], ['WEST'], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['CARE'], ['WEST'], ['NORTH'], ['CARE'], ['CARE'], ['WATER'], ['NORTH'], ['NORTH'], ['EAST'], ['WEST'], ['NORTH'], ['WATER'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['WEST'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['NORTH'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['EAST'], ['WATER'], ['FERTILIZE', 'FERTILIZER'], ['WATER'], ['WATER'], ['NORTH'], ['WEST'], ['FERTILIZE', 'FERTILIZER'], ['NORTH']], 'market': []}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['EAST'], ['FERTILIZE', 'FERTILIZER'], ['NORTH'], ['EAST'], ['SOUTH'], ['HARVEST'], ['WATER'], ['EAST'], ['SOUTH'], ['NORTH'], ['WATER'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['FEED', 'WHEAT'], ['SOUTH'], ['WATER'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['SOUTH'], ['PLANT', 'WHEAT'], ['WEST'], ['WATER'], ['WATER'], ['NORTH'], ['WEST'], ['EAST'], ['HARVEST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['CARE'], ['WATER'], ['EAST'], ['CARE'], ['CARE'], ['WATER'], ['WATER'], ['FERTILIZE', 'FERTILIZER'], ['NORTH'], ['SOUTH'], ['NORTH'], ['WATER'], ['FERTILIZE', 'FERTILIZER'], ['PLANT', 'WHEAT']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['COLLECT_FERTILIZER'], ['EAST'], ['EAST'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['EAST'], ['WEST'], ['WATER'], ['NORTH'], ['WATER'], ['WATER'], ['SOUTH'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['WEST'], ['WATER'], ['SOUTH'], ['WEST'], ['NORTH'], ['WATER'], ['NORTH'], ['WEST'], ['FERTILIZE', 'FERTILIZER'], ['EAST'], ['HARVEST'], ['SOUTH'], ['EAST'], ['WEST']], 'market': []}, {'farmer': ['WATER'], 'hands': [['WEST'], ['SOUTH'], ['WATER'], ['SOUTH'], ['HARVEST'], ['SOUTH'], ['NORTH'], ['FERTILIZE', 'FERTILIZER'], ['WATER'], ['WATER'], ['PLANT', 'WHEAT'], ['WATER'], ['FERTILIZE', 'FERTILIZER'], ['WATER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['SOUTH'], ['WATER'], ['HARVEST'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['WATER'], ['NORTH'], ['WATER'], ['NORTH'], ['SOUTH'], ['WATER'], ['WEST'], ['WATER'], ['HARVEST']], 'market': []}, {'farmer': ['WATER'], 'hands': [['SOUTH'], ['SOUTH'], ['PLANT', 'WHEAT'], ['CARE'], ['CARE'], ['WEST'], ['HARVEST'], ['NORTH'], ['FERTILIZE', 'FERTILIZER'], ['WATER'], ['WEST'], ['WATER'], ['SOUTH'], ['PLANT', 'WHEAT']], 'market': []}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['WATER'], ['WATER'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['WATER'], ['EAST'], ['WATER'], ['WATER'], ['WEST'], ['WATER'], ['WEST'], ['FERTILIZE', 'FERTILIZER'], ['WATER']], 'market': []}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['SOUTH'], ['EAST'], ['SOUTH'], ['EAST'], ['EAST'], ['EAST'], ['WEST'], ['NORTH'], ['WATER'], ['HARVEST'], ['WATER'], ['WATER'], ['PASS']], 'market': []}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['WATER'], ['WATER'], ['HARVEST'], ['SOUTH'], ['EAST'], ['SOUTH'], ['WATER'], ['FERTILIZE', 'FERTILIZER'], ['SOUTH'], ['PLANT', 'WHEAT'], ['SOUTH'], ['EAST'], ['PASS']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['CARE'], 'hands': [['WEST'], ['WEST'], ['HARVEST'], ['FEED', 'WHEAT'], ['HARVEST'], ['WEST'], ['WATER'], ['SOUTH'], ['WATER'], ['WATER'], ['WATER'], ['HARVEST'], ['FERTILIZE', 'FERTILIZER'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['WATER'], ['PLANT', 'WHEAT'], ['COLLECT_FERTILIZER'], ['FEED', 'WHEAT'], ['SOUTH'], ['PASS'], ['PASS'], ['WEST'], ['EAST'], ['WEST'], ['PASS'], ['WATER'], ['PASS']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['PASS'], ['WATER'], ['PASS'], ['CARE'], ['PASS'], ['PASS'], ['PASS'], ['SOUTH'], ['WATER'], ['WATER'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['WATER'], ['PASS'], ['PASS'], ['PASS'], ['COLLECT_FERTILIZER'], ['PASS'], ['PASS'], ['PASS'], ['FERTILIZE', 'FERTILIZER'], ['EAST'], ['HARVEST'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['WEST'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PLANT', 'WHEAT'], ['PASS'], ['PASS'], ['PASS']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['WEST'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['WATER'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'MILK', 18], ['SELL', 'WHEAT', 25], ['SELL', 'STRAWBERRY', 4], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['HIRE'], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['WATER'], ['NORTH'], ['PICKUP', 'WHEAT', 4], ['PICKUP', 'WHEAT', 4], ['WEST'], ['WEST'], ['WEST']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['FEED', 'WHEAT'], ['EAST'], ['PICKUP', 'FERTILIZER', 2], ['NORTH'], ['NORTH'], ['WEST'], ['WEST'], ['WEST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['CARE'], ['EAST'], ['NORTH'], ['FEED', 'WHEAT'], ['HARVEST'], ['SOUTH'], ['WEST'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['CARE'], ['FEED', 'WHEAT'], ['WATER'], ['WEST'], ['HARVEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['EAST'], ['NORTH'], ['NORTH'], ['COLLECT_FERTILIZER'], ['CARE'], ['WEST'], ['WEST'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['FEED', 'WHEAT'], ['NORTH'], ['NORTH'], ['NORTH'], ['COLLECT_FERTILIZER'], ['WATER'], ['NORTH'], ['EAST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['CARE'], ['HARVEST'], ['WATER'], ['FEED', 'WHEAT'], ['NORTH'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['COLLECT_FERTILIZER'], ['EAST'], ['EAST'], ['CARE'], ['HARVEST'], ['WATER'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['NORTH'], 'hands': [['WEST'], ['EAST'], ['WATER'], ['COLLECT_FERTILIZER'], ['FEED', 'WHEAT'], ['WEST'], ['WATER'], ['HARVEST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['NORTH'], ['NORTH'], ['EAST'], ['WEST'], ['CARE'], ['WATER'], ['NORTH'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['NORTH'], ['WATER'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['WATER'], ['HARVEST']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['NORTH'], ['WATER'], ['EAST'], ['FEED', 'WHEAT'], ['EAST'], ['WATER'], ['EAST'], ['WEST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['HARVEST'], ['WEST'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['EAST'], ['WATER'], ['HARVEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['HARVEST'], 'hands': [['EAST'], ['SOUTH'], ['SOUTH'], ['WEST'], ['FEED', 'WHEAT'], ['EAST'], ['EAST'], ['SOUTH']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['HARVEST'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['CARE'], ['NORTH'], ['WATER'], ['HARVEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['NORTH'], 'hands': [['EAST'], ['SOUTH'], ['SOUTH'], ['FEED', 'WHEAT'], ['COLLECT_FERTILIZER'], ['WATER'], ['EAST'], ['WEST']], 'market': []}, {'farmer': ['HARVEST'], 'hands': [['HARVEST'], ['SOUTH'], ['FERTILIZE', 'FERTILIZER'], ['WEST'], ['EAST'], ['EAST'], ['WATER'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['EAST'], ['CARE'], ['WATER'], ['WEST'], ['SOUTH'], ['WATER'], ['EAST'], ['HARVEST']], 'market': []}, {'farmer': ['PASS'], 'hands': [['HARVEST'], ['COLLECT_FERTILIZER'], ['EAST'], ['HARVEST'], ['FEED', 'WHEAT'], ['SOUTH'], ['WATER'], ['PLANT', 'WHEAT']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['EAST'], ['NORTH'], ['FERTILIZE', 'FERTILIZER'], ['PASS'], ['PASS'], ['WATER'], ['PASS'], ['EAST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['HARVEST'], ['NORTH'], ['WATER'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['EAST']], 'market': []}, {'farmer': ['WATER'], 'hands': [['NORTH'], ['HARVEST'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['EAST']], 'market': []}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'STRAWBERRY', 28], ['SELL', 'MILK', 9], ['SELL', 'FERTILIZER', 9], ['SELL', 'WHEAT', 4], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_ANIMAL', 'COW', 1], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 3], 'hands': [['PICKUP', 'WHEAT', 2], ['WATER'], ['EAST'], ['PICKUP', 'WHEAT', 3], ['PICKUP', 'WHEAT', 4], ['SOUTH'], ['WEST'], ['PICKUP', 'FERTILIZER', 4], ['NORTH'], ['WEST'], ['EAST'], ['WEST']], 'market': []}, {'farmer': ['HARVEST'], 'hands': [['NORTH'], ['WEST'], ['NORTH'], ['NORTH'], ['HARVEST'], ['WATER'], ['NORTH'], ['WEST'], ['NORTH'], ['WEST'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['FEED', 'WHEAT'], ['WATER'], ['NORTH'], ['HARVEST'], ['FEED', 'WHEAT'], ['SOUTH'], ['NORTH'], ['WEST'], ['NORTH'], ['SOUTH'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['CARE'], 'hands': [['CARE'], ['WEST'], ['NORTH'], ['FEED', 'WHEAT'], ['CARE'], ['WATER'], ['NORTH'], ['NORTH'], ['WATER'], ['SOUTH'], ['NORTH'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['NORTH'], ['CARE'], ['COLLECT_FERTILIZER'], ['WEST'], ['NORTH'], ['FERTILIZE', 'FERTILIZER'], ['EAST'], ['WATER'], ['NORTH'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['WEST'], ['WATER'], ['COLLECT_FERTILIZER'], ['EAST'], ['WATER'], ['WATER'], ['WATER'], ['SOUTH'], ['WEST'], ['NORTH'], ['WATER']], 'market': []}, {'farmer': ['HARVEST'], 'hands': [['FEED', 'WHEAT'], ['WATER'], ['EAST'], ['NORTH'], ['FEED', 'WHEAT'], ['SOUTH'], ['WEST'], ['WEST'], ['WATER'], ['WATER'], ['HARVEST'], ['WEST']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['CARE'], ['WEST'], ['WATER'], ['HARVEST'], ['CARE'], ['WATER'], ['WATER'], ['FERTILIZE', 'FERTILIZER'], ['EAST'], ['WEST'], ['WEST'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['CARE'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['EAST'], ['FEED', 'WHEAT'], ['COLLECT_FERTILIZER'], ['EAST'], ['WEST'], ['WATER'], ['WATER'], ['WATER'], ['WATER'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['WEST'], ['EAST'], ['WATER'], ['CARE'], ['NORTH'], ['WATER'], ['WATER'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['EAST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['SOUTH'], ['EAST'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['SOUTH'], ['WEST'], ['FERTILIZE', 'FERTILIZER'], ['WATER'], ['WATER'], ['EAST'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 5]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['NORTH'], ['WATER'], ['WATER'], ['WEST'], ['FEED', 'WHEAT'], ['WATER'], ['WATER'], ['WATER'], ['EAST'], ['EAST'], ['WATER'], ['EAST']], 'market': [['SELL', 'WHEAT', 5]]}, {'farmer': ['CARE'], 'hands': [['WATER'], ['EAST'], ['NORTH'], ['SOUTH'], ['CARE'], ['WEST'], ['WEST'], ['WEST'], ['WATER'], ['WATER'], ['EAST'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['WEST'], ['WATER'], ['WATER'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['WATER'], ['WATER'], ['FERTILIZE', 'FERTILIZER'], ['EAST'], ['EAST'], ['WATER'], ['EAST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['WEST'], ['EAST'], ['HARVEST'], ['FEED', 'WHEAT'], ['EAST'], ['WEST'], ['SOUTH'], ['WATER'], ['WATER'], ['WATER'], ['SOUTH'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['WEST'], 'hands': [['WEST'], ['WATER'], ['SOUTH'], ['CARE'], ['SOUTH'], ['WEST'], ['WATER'], ['NORTH'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['PASS']], 'market': [['SELL', 'WHEAT', 1], ['SELL', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['WEST'], ['PASS'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['WEST'], ['PASS'], ['NORTH'], ['HARVEST'], ['WATER'], ['WATER'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['WATER'], ['PASS'], ['WATER'], ['EAST'], ['FEED', 'WHEAT'], ['NORTH'], ['PASS'], ['NORTH'], ['WEST'], ['WEST'], ['HARVEST'], ['PASS']], 'market': [['SELL', 'WHEAT', 1]]}, {'farmer': ['WATER'], 'hands': [['PASS'], ['PASS'], ['HARVEST'], ['SOUTH'], ['CARE'], ['NORTH'], ['PASS'], ['PASS'], ['HARVEST'], ['WATER'], ['PASS'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PLACE', 'WOOL', 12], ['WEST'], ['PASS'], ['PASS'], ['PASS'], ['WEST'], ['WEST'], ['PASS'], ['PASS']], 'market': [['SELL', 'WHEAT', 2], ['SELL', 'WOOL', 12]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['WEST'], ['PASS'], ['PASS'], ['PASS'], ['COLLECT_FERTILIZER'], ['WATER'], ['PASS'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['DROP'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['SELL', 'MILK', 9], ['SELL', 'WHEAT', 2]]}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'WOOL', 16], ['SELL', 'STRAWBERRY', 12], ['SELL', 'FERTILIZER', 8], ['SELL', 'MILK', 3], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['WATER'], ['NORTH'], ['PICKUP', 'WHEAT', 4], ['PICKUP', 'WHEAT', 4], ['PICKUP', 'FERTILIZER', 6], ['NORTH'], ['WEST'], ['PICKUP', 'FERTILIZER', 1], ['PICKUP', 'FERTILIZER', 5], ['WEST'], ['WEST'], ['NORTH']], 'market': [['BUY_SEED', 'WHEAT', 4]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['FEED', 'WHEAT'], ['WEST'], ['PICKUP', 'FERTILIZER', 5], ['NORTH'], ['NORTH'], ['SOUTH'], ['NORTH'], ['NORTH'], ['EAST'], ['WEST'], ['WEST'], ['WEST'], ['NORTH']], 'market': []}, {'farmer': ['CARE'], 'hands': [['CARE'], ['WATER'], ['WEST'], ['FEED', 'WHEAT'], ['HARVEST'], ['FERTILIZE', 'FERTILIZER'], ['NORTH'], ['NORTH'], ['EAST'], ['WEST'], ['WEST'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['WEST'], ['CARE'], ['FEED', 'WHEAT'], ['WATER'], ['NORTH'], ['HARVEST'], ['EAST'], ['WEST'], ['WEST'], ['NORTH'], ['HARVEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['EAST'], ['WATER'], ['WEST'], ['COLLECT_FERTILIZER'], ['CARE'], ['SOUTH'], ['NORTH'], ['DIG'], ['WATER'], ['WEST'], ['WEST'], ['HARVEST'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['EAST'], ['WEST'], ['WEST'], ['NORTH'], ['COLLECT_FERTILIZER'], ['FERTILIZE', 'FERTILIZER'], ['WATER'], ['PLANT', 'WHEAT'], ['EAST'], ['SOUTH'], ['NORTH'], ['DIG'], ['HARVEST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['FEED', 'WHEAT'], ['WATER'], ['SOUTH'], ['FEED', 'WHEAT'], ['NORTH'], ['WATER'], ['HARVEST'], ['WATER'], ['WATER'], ['SOUTH'], ['NORTH'], ['PLANT', 'WHEAT'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['CARE'], ['WEST'], ['SOUTH'], ['CARE'], ['HARVEST'], ['WEST'], ['PLANT', 'WHEAT'], ['WEST'], ['WEST'], ['FERTILIZE', 'FERTILIZER'], ['WATER'], ['WATER'], ['HARVEST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['WATER'], ['COLLECT_FERTILIZER'], ['FEED', 'WHEAT'], ['FERTILIZE', 'FERTILIZER'], ['WATER'], ['SOUTH'], ['WEST'], ['WATER'], ['NORTH'], ['WEST'], ['EAST']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['NORTH'], ['SOUTH'], ['SOUTH'], ['WEST'], ['CARE'], ['WATER'], ['EAST'], ['HARVEST'], ['WEST'], ['SOUTH'], ['NORTH'], ['HARVEST'], ['HARVEST']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['WATER'], ['WATER'], ['FERTILIZE', 'FERTILIZER'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['EAST'], ['WEST'], ['NORTH'], ['FERTILIZE', 'FERTILIZER'], ['WATER'], ['DIG'], ['SOUTH']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['EAST'], ['EAST'], ['WATER'], ['FEED', 'WHEAT'], ['EAST'], ['FERTILIZE', 'FERTILIZER'], ['WATER'], ['SOUTH'], ['NORTH'], ['WATER'], ['EAST'], ['PLANT', 'WHEAT'], ['HARVEST']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['WATER'], ['EAST'], ['EAST'], ['CARE'], ['SOUTH'], ['WATER'], ['HARVEST'], ['HARVEST'], ['NORTH'], ['EAST'], ['WATER'], ['WATER'], ['WEST']], 'market': []}, {'farmer': ['WATER'], 'hands': [['EAST'], ['WATER'], ['FERTILIZE', 'FERTILIZER'], ['COLLECT_FERTILIZER'], ['FEED', 'WHEAT'], ['EAST'], ['PLANT', 'WHEAT'], ['WEST'], ['NORTH'], ['FERTILIZE', 'FERTILIZER'], ['EAST'], ['WEST'], ['HARVEST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['WATER'], ['EAST'], ['WATER'], ['WEST'], ['CARE'], ['FERTILIZE', 'FERTILIZER'], ['WATER'], ['HARVEST'], ['FERTILIZE', 'FERTILIZER'], ['WATER'], ['WATER'], ['WATER'], ['WEST']], 'market': []}, {'farmer': ['WATER'], 'hands': [['NORTH'], ['WATER'], ['SOUTH'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['WATER'], ['EAST'], ['NORTH'], ['WATER'], ['SOUTH'], ['EAST'], ['HARVEST'], ['HARVEST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['HARVEST'], ['SOUTH'], ['FERTILIZE', 'FERTILIZER'], ['HARVEST'], ['SOUTH'], ['SOUTH'], ['WATER'], ['NORTH'], ['EAST'], ['FERTILIZE', 'FERTILIZER'], ['WATER'], ['PLANT', 'WHEAT'], ['EAST']], 'market': []}, {'farmer': ['WATER'], 'hands': [['NORTH'], ['SOUTH'], ['WATER'], ['FEED', 'WHEAT'], ['HARVEST'], ['FERTILIZE', 'FERTILIZER'], ['HARVEST'], ['NORTH'], ['EAST'], ['WATER'], ['EAST'], ['WATER'], ['WEST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['HARVEST'], ['SOUTH'], ['SOUTH'], ['CARE'], ['WEST'], ['WATER'], ['PLANT', 'WHEAT'], ['NORTH'], ['EAST'], ['WEST'], ['WATER'], ['EAST'], ['SOUTH']], 'market': []}, {'farmer': ['WATER'], 'hands': [['NORTH'], ['WATER'], ['FERTILIZE', 'FERTILIZER'], ['COLLECT_FERTILIZER'], ['DROP'], ['PASS'], ['WATER'], ['WATER'], ['PASS'], ['FERTILIZE', 'FERTILIZER'], ['PASS'], ['SOUTH'], ['SOUTH']], 'market': [['SELL', 'MILK', 11]]}, {'farmer': ['PASS'], 'hands': [['HARVEST'], ['PASS'], ['WATER'], ['PASS'], ['EAST'], ['PASS'], ['EAST'], ['PASS'], ['PASS'], ['WATER'], ['PASS'], ['HARVEST'], ['WEST']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['EAST'], ['PASS'], ['CARE'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PLACE', 'STRAWBERRY', 14]], 'market': [['SELL', 'STRAWBERRY', 14]]}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'STRAWBERRY', 34], ['SELL', 'MILK', 3], ['SELL', 'WHEAT', 12], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_SEED', 'WHEAT', 45], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['WATER'], ['WEST'], ['PICKUP', 'WHEAT', 4], ['PICKUP', 'WHEAT', 4], ['WEST'], ['WEST'], ['WEST'], ['PICKUP', 'FERTILIZER', 5], ['WEST'], ['WEST'], ['NORTH'], ['PICKUP', 'FERTILIZER', 5], ['WEST']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['NORTH'], ['HARVEST'], ['WEST'], ['NORTH'], ['HARVEST'], ['WATER'], ['WEST'], ['WEST'], ['NORTH'], ['WEST'], ['WEST'], ['NORTH'], ['EAST'], ['SOUTH']], 'market': []}, {'farmer': ['CARE'], 'hands': [['FEED', 'WHEAT'], ['PLANT', 'WHEAT'], ['WEST'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['HARVEST'], ['WEST'], ['NORTH'], ['NORTH'], ['WATER'], ['NORTH'], ['NORTH'], ['EAST'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['CARE'], ['WATER'], ['WEST'], ['CARE'], ['CARE'], ['PLANT', 'WHEAT'], ['WEST'], ['WATER'], ['NORTH'], ['HARVEST'], ['NORTH'], ['WATER'], ['NORTH'], ['HARVEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['WEST'], 'hands': [['COLLECT_FERTILIZER'], ['DROP'], ['WATER'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['WATER'], ['WEST'], ['WEST'], ['FERTILIZE', 'FERTILIZER'], ['PLANT', 'WHEAT'], ['NORTH'], ['HARVEST'], ['NORTH'], ['PLANT', 'WHEAT']], 'market': [['SELL', 'MELON', 6]]}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['WEST'], ['HARVEST'], ['NORTH'], ['EAST'], ['EAST'], ['WATER'], ['WATER'], ['WATER'], ['WATER'], ['NORTH'], ['PLANT', 'WHEAT'], ['NORTH'], ['WATER']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['FEED', 'WHEAT'], ['WEST'], ['PLANT', 'WHEAT'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['DROP'], ['HARVEST'], ['SOUTH'], ['EAST'], ['EAST'], ['WATER'], ['WATER'], ['FERTILIZE', 'FERTILIZER'], ['EAST']], 'market': [['SELL', 'MELON', 6]]}, {'farmer': ['CARE'], 'hands': [['CARE'], ['NORTH'], ['WATER'], ['CARE'], ['CARE'], ['WEST'], ['PLANT', 'WHEAT'], ['WATER'], ['FERTILIZE', 'FERTILIZER'], ['EAST'], ['HARVEST'], ['SOUTH'], ['WATER'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 4]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['EAST'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['WEST'], ['WATER'], ['WEST'], ['WATER'], ['DROP'], ['PLANT', 'WHEAT'], ['SOUTH'], ['EAST'], ['DROP']], 'market': [['SELL', 'MELON', 12]]}, {'farmer': ['WEST'], 'hands': [['WEST'], ['NORTH'], ['EAST'], ['WEST'], ['NORTH'], ['WEST'], ['EAST'], ['WATER'], ['SOUTH'], ['WEST'], ['WATER'], ['SOUTH'], ['FERTILIZE', 'FERTILIZER'], ['EAST']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['WEST'], ['NORTH'], ['EAST'], ['SOUTH'], ['HARVEST'], ['WEST'], ['EAST'], ['NORTH'], ['FERTILIZE', 'FERTILIZER'], ['WEST'], ['EAST'], ['DROP'], ['WATER'], ['EAST']], 'market': [['SELL', 'MELON', 6]]}, {'farmer': ['NORTH'], 'hands': [['WEST'], ['WATER'], ['DROP'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['SOUTH'], ['EAST'], ['NORTH'], ['WATER'], ['SOUTH'], ['SOUTH'], ['NORTH'], ['EAST'], ['NORTH']], 'market': [['SELL', 'MELON', 6]]}, {'farmer': ['NORTH'], 'hands': [['WEST'], ['HARVEST'], ['SOUTH'], ['CARE'], ['CARE'], ['WATER'], ['EAST'], ['NORTH'], ['EAST'], ['WATER'], ['SOUTH'], ['NORTH'], ['FERTILIZE', 'FERTILIZER'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['WATER'], 'hands': [['WEST'], ['PLANT', 'WHEAT'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['DROP'], ['NORTH'], ['FERTILIZE', 'FERTILIZER'], ['WEST'], ['SOUTH'], ['NORTH'], ['WATER'], ['NORTH']], 'market': [['SELL', 'MELON', 6]]}, {'farmer': ['HARVEST'], 'hands': [['NORTH'], ['WATER'], ['SOUTH'], ['SOUTH'], ['EAST'], ['PLANT', 'WHEAT'], ['EAST'], ['WATER'], ['WATER'], ['WATER'], ['DROP'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'MELON', 6]]}, {'farmer': ['PLANT', 'WHEAT'], 'hands': [['WATER'], ['EAST'], ['HARVEST'], ['HARVEST'], ['SOUTH'], ['WATER'], ['EAST'], ['HARVEST'], ['EAST'], ['SOUTH'], ['WEST'], ['WATER'], ['FERTILIZE', 'FERTILIZER'], ['NORTH']], 'market': []}, {'farmer': ['WATER'], 'hands': [['HARVEST'], ['EAST'], ['WEST'], ['FEED', 'WHEAT'], ['HARVEST'], ['EAST'], ['EAST'], ['PLANT', 'WHEAT'], ['FERTILIZE', 'FERTILIZER'], ['HARVEST'], ['NORTH'], ['HARVEST'], ['WATER'], ['HARVEST']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['PLANT', 'WHEAT'], ['SOUTH'], ['HARVEST'], ['CARE'], ['WEST'], ['EAST'], ['NORTH'], ['WATER'], ['WATER'], ['WEST'], ['NORTH'], ['PLANT', 'WHEAT'], ['SOUTH'], ['WEST']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['WATER'], ['SOUTH'], ['WEST'], ['COLLECT_FERTILIZER'], ['WEST'], ['EAST'], ['NORTH'], ['EAST'], ['EAST'], ['HARVEST'], ['NORTH'], ['WATER'], ['SOUTH'], ['WEST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['EAST'], ['SOUTH'], ['HARVEST'], ['PASS'], ['DROP'], ['EAST'], ['WATER'], ['WATER'], ['WATER'], ['SOUTH'], ['NORTH'], ['WEST'], ['FERTILIZE', 'FERTILIZER'], ['WEST']], 'market': [['SELL', 'MILK', 9], ['SELL', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['EAST'], ['DROP'], ['SOUTH'], ['PASS'], ['EAST'], ['NORTH'], ['HARVEST'], ['HARVEST'], ['SOUTH'], ['HARVEST'], ['WATER'], ['WEST'], ['PASS'], ['WEST']], 'market': [['SELL', 'MELON', 6]]}, {'farmer': ['SOUTH'], 'hands': [['EAST'], ['PASS'], ['EAST'], ['PASS'], ['EAST'], ['DROP'], ['PLANT', 'WHEAT'], ['PASS'], ['WATER'], ['EAST'], ['HARVEST'], ['WEST'], ['PASS'], ['WATER']], 'market': [['SELL', 'MELON', 6], ['BUY_SEED', 'WHEAT', 2]]}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'MELON', 12], ['SELL', 'STRAWBERRY', 16], ['SELL', 'WHEAT', 20], ['SELL', 'MILK', 3], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_SEED', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 3], 'hands': [['PICKUP', 'COW', 1], ['PICKUP', 'FERTILIZER', 1], ['NORTH'], ['PICKUP', 'WHEAT', 3], ['PICKUP', 'WHEAT', 4], ['SOUTH'], ['WEST'], ['PICKUP', 'COW', 1], ['PICKUP', 'FERTILIZER', 2], ['WEST'], ['WEST'], ['WEST']], 'market': []}, {'farmer': ['HARVEST'], 'hands': [['PICKUP', 'WHEAT', 3], ['SOUTH'], ['NORTH'], ['NORTH'], ['NORTH'], ['WATER'], ['WEST'], ['PICKUP', 'WHEAT', 1], ['EAST'], ['WEST'], ['WEST'], ['NORTH']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['FEED', 'WHEAT'], ['SOUTH'], ['NORTH'], ['NORTH'], ['HARVEST'], ['SOUTH'], ['WEST'], ['WEST'], ['EAST'], ['SOUTH'], ['WEST'], ['NORTH']], 'market': []}, {'farmer': ['CARE'], 'hands': [['CARE'], ['SOUTH'], ['NORTH'], ['HARVEST'], ['FEED', 'WHEAT'], ['WATER'], ['SOUTH'], ['NORTH'], ['EAST'], ['WATER'], ['NORTH'], ['WATER']], 'market': [['BUY_SEED', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['NORTH'], ['FEED', 'WHEAT'], ['CARE'], ['WEST'], ['SOUTH'], ['NORTH'], ['FERTILIZE', 'FERTILIZER'], ['HARVEST'], ['NORTH'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['NORTH'], 'hands': [['EAST'], ['HARVEST'], ['WATER'], ['CARE'], ['COLLECT_FERTILIZER'], ['WATER'], ['SOUTH'], ['NORTH'], ['WATER'], ['PLANT', 'WHEAT'], ['NORTH'], ['WATER']], 'market': [['BUY_SEED', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['HARVEST'], 'hands': [['FEED', 'WHEAT'], ['WEST'], ['EAST'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['WATER'], ['NORTH'], ['HARVEST'], ['WATER'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['CARE'], ['WATER'], ['WATER'], ['WEST'], ['HARVEST'], ['WATER'], ['HARVEST'], ['BUILD_PASTURE'], ['NORTH'], ['NORTH'], ['NORTH'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['CARE'], 'hands': [['COLLECT_FERTILIZER'], ['HARVEST'], ['EAST'], ['SOUTH'], ['FEED', 'WHEAT'], ['WEST'], ['WEST'], ['PLACE', 'COW'], ['DIG'], ['NORTH'], ['DIG'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['EAST'], ['SOUTH'], ['WATER'], ['HARVEST'], ['CARE'], ['WATER'], ['WATER'], ['FEED', 'WHEAT'], ['PLANT', 'WHEAT'], ['NORTH'], ['PLANT', 'WHEAT'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['FERTILIZE', 'FERTILIZER'], ['EAST'], ['FEED', 'WHEAT'], ['COLLECT_FERTILIZER'], ['WEST'], ['HARVEST'], ['CARE'], ['WATER'], ['HARVEST'], ['WATER'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['DIG'], ['WATER'], ['WATER'], ['CARE'], ['EAST'], ['WATER'], ['WEST'], ['EAST'], ['EAST'], ['DIG'], ['WEST'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['BUILD_PASTURE'], ['EAST'], ['EAST'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['EAST'], ['WATER'], ['EAST'], ['SOUTH'], ['PLANT', 'WHEAT'], ['PLANT', 'WHEAT'], ['HARVEST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['PLACE', 'COW'], ['WATER'], ['HARVEST'], ['WEST'], ['WEST'], ['NORTH'], ['SOUTH'], ['SOUTH'], ['FERTILIZE', 'FERTILIZER'], ['WATER'], ['WATER'], ['PLANT', 'WHEAT']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['FEED', 'WHEAT'], ['HARVEST'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['WATER'], ['WATER'], ['HARVEST'], ['WATER'], ['WEST'], ['WEST'], ['WATER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['CARE'], ['WEST'], ['HARVEST'], ['HARVEST'], ['DROP'], ['HARVEST'], ['HARVEST'], ['EAST'], ['HARVEST'], ['SOUTH'], ['SOUTH'], ['EAST']], 'market': [['SELL', 'MILK', 6]]}, {'farmer': ['CARE'], 'hands': [['WEST'], ['EAST'], ['WEST'], ['EAST'], ['PICKUP', 'WHEAT', 2], ['PLANT', 'WHEAT'], ['EAST'], ['HARVEST'], ['NORTH'], ['HARVEST'], ['SOUTH'], ['HARVEST']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['CARE'], ['NORTH'], ['HARVEST'], ['EAST'], ['EAST'], ['WATER'], ['WATER'], ['EAST'], ['DIG'], ['EAST'], ['PASS'], ['DIG']], 'market': []}, {'farmer': ['PASS'], 'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['SOUTH'], ['DROP'], ['NORTH'], ['PASS'], ['HARVEST'], ['HARVEST'], ['PLANT', 'WHEAT'], ['EAST'], ['PASS'], ['PLANT', 'WHEAT']], 'market': [['SELL', 'WOOL', 8], ['SELL', 'MILK', 3]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['NORTH'], ['WEST'], ['WEST'], ['FEED', 'WHEAT'], ['PASS'], ['EAST'], ['SOUTH'], ['WATER'], ['EAST'], ['PASS'], ['WATER']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['NORTH'], ['WEST'], ['WEST'], ['EAST'], ['PASS'], ['WATER'], ['HARVEST'], ['PASS'], ['DROP'], ['PASS'], ['PASS']], 'market': [['SELL', 'MELON', 6], ['SELL', 'STRAWBERRY', 4]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['DROP'], ['WEST'], ['PASS'], ['SOUTH'], ['PASS'], ['HARVEST'], ['WEST'], ['PASS'], ['WEST'], ['PASS'], ['PASS']], 'market': [['SELL', 'STRAWBERRY', 6]]}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'STRAWBERRY', 30], ['SELL', 'FERTILIZER', 5], ['SELL', 'WOOL', 8], ['SELL', 'WHEAT', 1], ['SELL', 'MELON', 6], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 3], 'hands': [['PICKUP', 'WHEAT', 2], ['WATER'], ['EAST'], ['PICKUP', 'WHEAT', 4], ['PICKUP', 'WHEAT', 4], ['SOUTH'], ['EAST'], ['NORTH'], ['NORTH'], ['WEST'], ['WEST']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['NORTH'], ['WEST'], ['NORTH'], ['WEST'], ['HARVEST'], ['HARVEST'], ['EAST'], ['NORTH'], ['NORTH'], ['WEST'], ['NORTH']], 'market': []}, {'farmer': ['CARE'], 'hands': [['FEED', 'WHEAT'], ['WATER'], ['NORTH'], ['HARVEST'], ['FEED', 'WHEAT'], ['SOUTH'], ['EAST'], ['NORTH'], ['NORTH'], ['SOUTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['CARE'], ['WEST'], ['NORTH'], ['FEED', 'WHEAT'], ['CARE'], ['HARVEST'], ['NORTH'], ['WATER'], ['WATER'], ['SOUTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['NORTH'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['WATER'], ['CARE'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['HARVEST'], ['WEST'], ['EAST'], ['HARVEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['EAST'], ['WEST'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['NORTH'], ['HARVEST'], ['EAST'], ['SOUTH'], ['WATER'], ['WEST'], ['NORTH']], 'market': []}, {'farmer': ['CARE'], 'hands': [['SOUTH'], ['WATER'], ['EAST'], ['NORTH'], ['NORTH'], ['SOUTH'], ['HARVEST'], ['WATER'], ['EAST'], ['HARVEST'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['FEED', 'WHEAT'], ['WEST'], ['NORTH'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['HARVEST'], ['WEST'], ['WEST'], ['SOUTH'], ['WEST'], ['WEST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['CARE'], ['WATER'], ['WATER'], ['CARE'], ['CARE'], ['WEST'], ['WEST'], ['WATER'], ['WATER'], ['HARVEST'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['WEST'], 'hands': [['COLLECT_FERTILIZER'], ['SOUTH'], ['EAST'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['WEST'], ['WEST'], ['SOUTH'], ['WATER']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['EAST'], ['WATER'], ['WATER'], ['EAST'], ['EAST'], ['NORTH'], ['WEST'], ['WATER'], ['NORTH'], ['HARVEST'], ['WEST']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['EAST'], ['EAST'], ['EAST'], ['NORTH'], ['SOUTH'], ['NORTH'], ['NORTH'], ['WEST'], ['NORTH'], ['EAST'], ['WATER']], 'market': []}, {'farmer': ['CARE'], 'hands': [['NORTH'], ['EAST'], ['WATER'], ['FEED', 'WHEAT'], ['HARVEST'], ['WATER'], ['NORTH'], ['WATER'], ['HARVEST'], ['HARVEST'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['NORTH'], ['EAST'], ['NORTH'], ['CARE'], ['FEED', 'WHEAT'], ['SOUTH'], ['NORTH'], ['EAST'], ['DIG'], ['EAST'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['WATER'], ['SOUTH'], ['WATER'], ['COLLECT_FERTILIZER'], ['CARE'], ['HARVEST'], ['NORTH'], ['SOUTH'], ['PLANT', 'WHEAT'], ['HARVEST'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['NORTH'], 'hands': [['EAST'], ['SOUTH'], ['SOUTH'], ['WEST'], ['COLLECT_FERTILIZER'], ['EAST'], ['WATER'], ['SOUTH'], ['WATER'], ['SOUTH'], ['WATER']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['WATER'], ['HARVEST'], ['SOUTH'], ['NORTH'], ['EAST'], ['NORTH'], ['WEST'], ['DIG'], ['EAST'], ['HARVEST'], ['NORTH']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['WEST'], ['SOUTH'], ['HARVEST'], ['NORTH'], ['FEED', 'WHEAT'], ['NORTH'], ['WEST'], ['PLANT', 'WHEAT'], ['WATER'], ['EAST'], ['WATER']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['HARVEST'], ['HARVEST'], ['WEST'], ['FEED', 'WHEAT'], ['CARE'], ['DROP'], ['CARE'], ['WATER'], ['EAST'], ['EAST'], ['PASS']], 'market': [['SELL', 'STRAWBERRY', 10], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['WEST'], ['PASS'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['PASS'], ['PASS'], ['PASS'], ['WATER'], ['NORTH'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['WEST'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['NORTH'], ['PASS']], 'market': [['SELL', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['HARVEST'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['NORTH'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'STRAWBERRY', 33], ['SELL', 'MILK', 9], ['SELL', 'FERTILIZER', 13], ['SELL', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 3], 'hands': [['PICKUP', 'WHEAT', 3], ['PICKUP', 'FERTILIZER', 5], ['WEST'], ['PICKUP', 'WHEAT', 4], ['PICKUP', 'WHEAT', 3], ['PICKUP', 'FERTILIZER', 6], ['EAST'], ['WEST'], ['EAST'], ['PICKUP', 'FERTILIZER', 4], ['EAST'], ['WEST'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['FEED', 'WHEAT'], ['WEST'], ['WEST'], ['NORTH'], ['NORTH'], ['SOUTH'], ['EAST'], ['WEST'], ['EAST'], ['WEST'], ['EAST'], ['NORTH'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['CARE'], ['WEST'], ['WEST'], ['FEED', 'WHEAT'], ['HARVEST'], ['FERTILIZE', 'FERTILIZER'], ['NORTH'], ['NORTH'], ['EAST'], ['SOUTH'], ['NORTH'], ['NORTH'], ['NORTH'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['COLLECT_FERTILIZER'], ['SOUTH'], ['NORTH'], ['CARE'], ['FEED', 'WHEAT'], ['WATER'], ['NORTH'], ['WATER'], ['WATER'], ['SOUTH'], ['NORTH'], ['WATER'], ['HARVEST'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['EAST'], ['SOUTH'], ['NORTH'], ['COLLECT_FERTILIZER'], ['CARE'], ['SOUTH'], ['NORTH'], ['WEST'], ['NORTH'], ['SOUTH'], ['NORTH'], ['HARVEST'], ['DIG'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['NORTH'], ['FERTILIZE', 'FERTILIZER'], ['NORTH'], ['NORTH'], ['COLLECT_FERTILIZER'], ['FERTILIZE', 'FERTILIZER'], ['NORTH'], ['WATER'], ['WATER'], ['SOUTH'], ['NORTH'], ['PLANT', 'WHEAT'], ['PLANT', 'WHEAT'], ['WATER']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['CARE'], 'hands': [['FEED', 'WHEAT'], ['WATER'], ['WATER'], ['FEED', 'WHEAT'], ['NORTH'], ['WATER'], ['HARVEST'], ['WEST'], ['EAST'], ['WATER'], ['NORTH'], ['WATER'], ['WATER'], ['NORTH']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['CARE'], ['WEST'], ['HARVEST'], ['CARE'], ['HARVEST'], ['WEST'], ['DIG'], ['WATER'], ['WATER'], ['WEST'], ['WATER'], ['NORTH'], ['EAST'], ['WATER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['COLLECT_FERTILIZER'], ['FERTILIZE', 'FERTILIZER'], ['PLANT', 'WHEAT'], ['COLLECT_FERTILIZER'], ['FEED', 'WHEAT'], ['FERTILIZE', 'FERTILIZER'], ['PLANT', 'WHEAT'], ['EAST'], ['SOUTH'], ['WEST'], ['HARVEST'], ['WATER'], ['SOUTH'], ['WEST']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['EAST'], ['WATER'], ['WATER'], ['WEST'], ['CARE'], ['WATER'], ['WATER'], ['NORTH'], ['WATER'], ['FERTILIZE', 'FERTILIZER'], ['PLANT', 'WHEAT'], ['WEST'], ['HARVEST'], ['WATER']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['CARE'], 'hands': [['FEED', 'WHEAT'], ['SOUTH'], ['WEST'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['EAST'], ['NORTH'], ['NORTH'], ['WATER'], ['WATER'], ['WATER'], ['DIG'], ['SOUTH']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['CARE'], ['FERTILIZE', 'FERTILIZER'], ['WATER'], ['FEED', 'WHEAT'], ['EAST'], ['FERTILIZE', 'FERTILIZER'], ['HARVEST'], ['NORTH'], ['NORTH'], ['WEST'], ['EAST'], ['EAST'], ['PLANT', 'WHEAT'], ['WATER']], 'market': []}, {'farmer': ['EAST'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['HARVEST'], ['CARE'], ['SOUTH'], ['WATER'], ['DIG'], ['WATER'], ['NORTH'], ['FERTILIZE', 'FERTILIZER'], ['WATER'], ['EAST'], ['WATER'], ['EAST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['WEST'], ['EAST'], ['PLANT', 'WHEAT'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['EAST'], ['PLANT', 'WHEAT'], ['EAST'], ['NORTH'], ['WATER'], ['HARVEST'], ['WATER'], ['EAST'], ['EAST']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['WEST'], ['FERTILIZE', 'FERTILIZER'], ['WATER'], ['NORTH'], ['HARVEST'], ['FERTILIZE', 'FERTILIZER'], ['WATER'], ['WATER'], ['HARVEST'], ['HARVEST'], ['PLANT', 'WHEAT'], ['NORTH'], ['HARVEST'], ['NORTH']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['WATER'], 'hands': [['WEST'], ['WATER'], ['WEST'], ['NORTH'], ['FEED', 'WHEAT'], ['WATER'], ['EAST'], ['WEST'], ['DIG'], ['NORTH'], ['WATER'], ['WATER'], ['DIG'], ['WATER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['WEST'], ['SOUTH'], ['WATER'], ['NORTH'], ['CARE'], ['SOUTH'], ['SOUTH'], ['WEST'], ['PLANT', 'WHEAT'], ['FERTILIZE', 'FERTILIZER'], ['EAST'], ['EAST'], ['PLANT', 'WHEAT'], ['EAST']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['WEST'], ['FERTILIZE', 'FERTILIZER'], ['HARVEST'], ['FEED', 'WHEAT'], ['COLLECT_FERTILIZER'], ['FERTILIZE', 'FERTILIZER'], ['HARVEST'], ['WATER'], ['WATER'], ['WATER'], ['SOUTH'], ['WATER'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['WATER'], 'hands': [['WEST'], ['WATER'], ['PLANT', 'WHEAT'], ['CARE'], ['PASS'], ['WATER'], ['DIG'], ['EAST'], ['WEST'], ['NORTH'], ['HARVEST'], ['HARVEST'], ['WEST'], ['SOUTH']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['WEST'], ['PASS'], ['WATER'], ['COLLECT_FERTILIZER'], ['WEST'], ['PASS'], ['PLANT', 'WHEAT'], ['SOUTH'], ['SOUTH'], ['FERTILIZE', 'FERTILIZER'], ['DIG'], ['PLANT', 'WHEAT'], ['NORTH'], ['PASS']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['PASS'], ['NORTH'], ['PASS'], ['DROP'], ['PASS'], ['WATER'], ['WATER'], ['SOUTH'], ['WATER'], ['PLANT', 'WHEAT'], ['WATER'], ['HARVEST'], ['PASS']], 'market': [['SELL', 'MILK', 11]]}, {'farmer': ['PASS'], 'hands': [['DIG'], ['PASS'], ['WATER'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['HARVEST'], ['PASS'], ['WATER'], ['EAST'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'STRAWBERRY', 22], ['SELL', 'WHEAT', 20], ['SELL', 'FERTILIZER', 2], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['SELL', 'FERTILIZER', 2], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 1], 'hands': [['PICKUP', 'WHEAT', 2], ['WATER'], ['WEST'], ['PICKUP', 'WHEAT', 3], ['PICKUP', 'WHEAT', 4], ['SOUTH'], ['EAST'], ['PICKUP', 'WHEAT', 3], ['WEST'], ['WEST'], ['WEST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['HARVEST'], ['NORTH'], ['NORTH'], ['HARVEST'], ['HARVEST'], ['NORTH'], ['HARVEST'], ['WEST'], ['WEST'], ['NORTH']], 'market': []}, {'farmer': ['WEST'], 'hands': [['FEED', 'WHEAT'], ['PLANT', 'WHEAT'], ['NORTH'], ['HARVEST'], ['FEED', 'WHEAT'], ['SOUTH'], ['NORTH'], ['FEED', 'WHEAT'], ['WEST'], ['WATER'], ['NORTH']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['CARE'], ['WATER'], ['NORTH'], ['FEED', 'WHEAT'], ['CARE'], ['HARVEST'], ['NORTH'], ['CARE'], ['WEST'], ['HARVEST'], ['NORTH']], 'market': []}, {'farmer': ['CARE'], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['NORTH'], ['CARE'], ['COLLECT_FERTILIZER'], ['WEST'], ['NORTH'], ['COLLECT_FERTILIZER'], ['WATER'], ['PLANT', 'WHEAT'], ['NORTH']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['EAST'], ['WATER'], ['WATER'], ['COLLECT_FERTILIZER'], ['NORTH'], ['HARVEST'], ['NORTH'], ['WEST'], ['NORTH'], ['WATER'], ['NORTH']], 'market': []}, {'farmer': ['WEST'], 'hands': [['SOUTH'], ['HARVEST'], ['HARVEST'], ['NORTH'], ['NORTH'], ['WEST'], ['WATER'], ['HARVEST'], ['NORTH'], ['WEST'], ['WATER']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['FEED', 'WHEAT'], ['PLANT', 'WHEAT'], ['PLANT', 'WHEAT'], ['HARVEST'], ['FEED', 'WHEAT'], ['HARVEST'], ['EAST'], ['FEED', 'WHEAT'], ['NORTH'], ['WATER'], ['HARVEST']], 'market': [['BUY_SEED', 'WHEAT', 2]]}, {'farmer': ['SOUTH'], 'hands': [['CARE'], ['WATER'], ['WATER'], ['FEED', 'WHEAT'], ['CARE'], ['EAST'], ['EAST'], ['CARE'], ['WATER'], ['HARVEST'], ['PLANT', 'WHEAT']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['COLLECT_FERTILIZER'], ['SOUTH'], ['WEST'], ['CARE'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['PLANT', 'WHEAT'], ['WATER']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['HARVEST'], 'hands': [['WEST'], ['WATER'], ['WEST'], ['COLLECT_FERTILIZER'], ['EAST'], ['HARVEST'], ['SOUTH'], ['EAST'], ['PLANT', 'WHEAT'], ['WATER'], ['WEST']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['WEST'], ['HARVEST'], ['NORTH'], ['WEST'], ['WEST'], ['EAST'], ['SOUTH'], ['DROP'], ['WATER'], ['WEST'], ['SOUTH']], 'market': [['SELL', 'WOOL', 4], ['SELL', 'MILK', 3], ['SELL', 'FERTILIZER', 2]]}, {'farmer': ['HARVEST'], 'hands': [['WEST'], ['PLANT', 'WHEAT'], ['WATER'], ['SOUTH'], ['SOUTH'], ['HARVEST'], ['SOUTH'], ['PICKUP', 'WHEAT', 1], ['WEST'], ['WATER'], ['WATER']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['WEST'], ['EAST'], ['WEST'], ['HARVEST'], ['SOUTH'], ['SOUTH'], ['HARVEST'], ['WEST'], ['WATER'], ['HARVEST'], ['HARVEST']], 'market': []}, {'farmer': ['HARVEST'], 'hands': [['SOUTH'], ['NORTH'], ['WATER'], ['FEED', 'WHEAT'], ['DROP'], ['HARVEST'], ['DIG'], ['NORTH'], ['HARVEST'], ['EAST'], ['PLANT', 'WHEAT']], 'market': [['SELL', 'MILK', 3], ['SELL', 'FERTILIZER', 2], ['BUY_SEED', 'WHEAT', 2]]}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['DROP'], ['SOUTH'], ['CARE'], ['PICKUP', 'WHEAT', 2], ['PASS'], ['PLANT', 'WHEAT'], ['NORTH'], ['PLANT', 'WHEAT'], ['EAST'], ['WATER']], 'market': [['SELL', 'FERTILIZER', 1], ['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['HARVEST'], 'hands': [['WATER'], ['WEST'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['EAST'], ['NORTH'], ['WATER'], ['NORTH'], ['WATER'], ['EAST'], ['WEST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['WEST'], ['SOUTH'], ['SOUTH'], ['WEST'], ['NORTH'], ['NORTH'], ['EAST'], ['NORTH'], ['NORTH'], ['EAST'], ['WATER']], 'market': []}, {'farmer': ['HARVEST'], 'hands': [['WATER'], ['WATER'], ['WATER'], ['WATER'], ['HARVEST'], ['NORTH'], ['HARVEST'], ['FEED', 'WHEAT'], ['WATER'], ['DROP'], ['HARVEST']], 'market': [['SELL', 'WHEAT', 12]]}, {'farmer': ['SOUTH'], 'hands': [['PASS'], ['WEST'], ['WEST'], ['WEST'], ['WEST'], ['NORTH'], ['DIG'], ['CARE'], ['HARVEST'], ['WEST'], ['EAST']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['HARVEST'], 'hands': [['PASS'], ['WEST'], ['WATER'], ['WEST'], ['SOUTH'], ['DROP'], ['PLANT', 'WHEAT'], ['COLLECT_FERTILIZER'], ['PLANT', 'WHEAT'], ['WEST'], ['EAST']], 'market': [['SELL', 'STRAWBERRY', 14], ['SELL', 'FERTILIZER', 1], ['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['PASS'], ['WEST'], ['PASS'], ['SOUTH'], ['DROP'], ['PASS'], ['WATER'], ['PASS'], ['WATER'], ['WEST'], ['SOUTH']], 'market': [['SELL', 'MILK', 3], ['SELL', 'WHEAT', 2]]}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'STRAWBERRY', 16], ['SELL', 'WHEAT', 28], ['SELL', 'FERTILIZER', 7], ['SELL', 'WOOL', 12], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 4], 'hands': [['PICKUP', 'WHEAT', 3], ['PICKUP', 'FERTILIZER', 1], ['WEST'], ['PICKUP', 'WHEAT', 3], ['PICKUP', 'WHEAT', 3], ['SOUTH'], ['EAST'], ['WEST'], ['NORTH'], ['WEST'], ['WEST'], ['WEST']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['FEED', 'WHEAT'], ['WEST'], ['WEST'], ['NORTH'], ['NORTH'], ['WATER'], ['EAST'], ['NORTH'], ['NORTH'], ['WEST'], ['WEST'], ['WEST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['CARE'], ['WEST'], ['WEST'], ['NORTH'], ['HARVEST'], ['SOUTH'], ['EAST'], ['NORTH'], ['NORTH'], ['SOUTH'], ['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['SOUTH'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['WATER'], ['NORTH'], ['WATER'], ['WATER'], ['WATER'], ['WEST'], ['WATER']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['NORTH'], 'hands': [['EAST'], ['WEST'], ['SOUTH'], ['CARE'], ['CARE'], ['WEST'], ['NORTH'], ['WEST'], ['NORTH'], ['HARVEST'], ['WEST'], ['HARVEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['NORTH'], ['SOUTH'], ['WATER'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['WATER'], ['WATER'], ['WATER'], ['WATER'], ['PLANT', 'WHEAT'], ['NORTH'], ['PLANT', 'WHEAT']], 'market': [['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['CARE'], 'hands': [['FEED', 'WHEAT'], ['DIG'], ['WEST'], ['WEST'], ['NORTH'], ['SOUTH'], ['HARVEST'], ['WEST'], ['EAST'], ['WATER'], ['NORTH'], ['WATER']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['CARE'], ['PLANT', 'WHEAT'], ['WATER'], ['NORTH'], ['HARVEST'], ['WATER'], ['PLANT', 'WHEAT'], ['WATER'], ['EAST'], ['WEST'], ['WATER'], ['WEST']], 'market': [['NOOP']]}, {'farmer': ['WEST'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['WEST'], ['NORTH'], ['FEED', 'WHEAT'], ['EAST'], ['WATER'], ['WEST'], ['WATER'], ['WATER'], ['HARVEST'], ['WATER']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['EAST'], ['SOUTH'], ['WATER'], ['FEED', 'WHEAT'], ['CARE'], ['WATER'], ['EAST'], ['WATER'], ['SOUTH'], ['HARVEST'], ['PLANT', 'WHEAT'], ['HARVEST']], 'market': [['NOOP']]}, {'farmer': ['CARE'], 'hands': [['FEED', 'WHEAT'], ['SOUTH'], ['SOUTH'], ['CARE'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['WATER'], ['EAST'], ['WATER'], ['PLANT', 'WHEAT'], ['WATER'], ['PLANT', 'WHEAT']], 'market': [['NOOP']]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['CARE'], ['SOUTH'], ['WATER'], ['COLLECT_FERTILIZER'], ['EAST'], ['WATER'], ['HARVEST'], ['NORTH'], ['EAST'], ['WATER'], ['EAST'], ['WATER']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['EAST'], ['WEST'], ['SOUTH'], ['WEST'], ['PLANT', 'WHEAT'], ['NORTH'], ['WATER'], ['WEST'], ['EAST'], ['SOUTH']], 'market': [['NOOP']]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['WEST'], ['HARVEST'], ['WATER'], ['SOUTH'], ['SOUTH'], ['WATER'], ['WATER'], ['WATER'], ['WEST'], ['NORTH'], ['NORTH'], ['WATER']], 'market': []}, {'farmer': ['CARE'], 'hands': [['NORTH'], ['EAST'], ['EAST'], ['SOUTH'], ['HARVEST'], ['WEST'], ['NORTH'], ['HARVEST'], ['SOUTH'], ['PLANT', 'WHEAT'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['NORTH'], ['EAST'], ['WATER'], ['SOUTH'], ['FEED', 'WHEAT'], ['WATER'], ['WATER'], ['PLANT', 'WHEAT'], ['WATER'], ['WATER'], ['NORTH'], ['PLANT', 'WHEAT']], 'market': [['NOOP']]}, {'farmer': ['WEST'], 'hands': [['DIG'], ['EAST'], ['WEST'], ['SOUTH'], ['CARE'], ['WEST'], ['NORTH'], ['WATER'], ['WEST'], ['EAST'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['CARE'], 'hands': [['PLANT', 'WHEAT'], ['FERTILIZE', 'FERTILIZER'], ['SOUTH'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['WATER'], ['WATER'], ['PASS'], ['WATER'], ['EAST'], ['HARVEST'], ['PASS']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['WATER'], ['HARVEST'], ['HARVEST'], ['FEED', 'WHEAT'], ['WEST'], ['PASS'], ['NORTH'], ['PASS'], ['NORTH'], ['PASS'], ['PLANT', 'WHEAT'], ['PASS']], 'market': [['NOOP']]}, {'farmer': ['PASS'], 'hands': [['EAST'], ['PASS'], ['PASS'], ['PASS'], ['DROP'], ['PASS'], ['WATER'], ['PASS'], ['NORTH'], ['PASS'], ['WATER'], ['PASS']], 'market': [['SELL', 'MILK', 9], ['SELL', 'FERTILIZER', 2]]}, {'farmer': ['PASS'], 'hands': [['EAST'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['WEST'], ['PASS'], ['WATER'], ['PASS'], ['SOUTH'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['WATER'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'STRAWBERRY', 6], ['SELL', 'FERTILIZER', 10], ['SELL', 'MILK', 5], ['SELL', 'WHEAT', 26], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_SEED', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 3], 'hands': [['PICKUP', 'WHEAT', 2], ['WATER'], ['WEST'], ['PICKUP', 'WHEAT', 4], ['PICKUP', 'WHEAT', 4], ['SOUTH'], ['WEST'], ['NORTH'], ['EAST'], ['WEST'], ['WEST'], ['WEST'], ['NORTH']], 'market': [['BUY_SEED', 'WHEAT', 2]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['NORTH'], ['WEST'], ['WEST'], ['WEST'], ['HARVEST'], ['HARVEST'], ['WEST'], ['NORTH'], ['EAST'], ['SOUTH'], ['WEST'], ['WEST'], ['NORTH']], 'market': []}, {'farmer': ['CARE'], 'hands': [['FEED', 'WHEAT'], ['WATER'], ['WEST'], ['HARVEST'], ['FEED', 'WHEAT'], ['DIG'], ['WEST'], ['NORTH'], ['EAST'], ['SOUTH'], ['WEST'], ['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['CARE'], ['WEST'], ['SOUTH'], ['FEED', 'WHEAT'], ['CARE'], ['PLANT', 'WHEAT'], ['WEST'], ['WATER'], ['WATER'], ['HARVEST'], ['WEST'], ['WATER'], ['WATER']], 'market': [['SELL', 'FERTILIZER', 1], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['NORTH'], 'hands': [['COLLECT_FERTILIZER'], ['WATER'], ['SOUTH'], ['CARE'], ['COLLECT_FERTILIZER'], ['WATER'], ['SOUTH'], ['NORTH'], ['EAST'], ['DIG'], ['WEST'], ['HARVEST'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['EAST'], ['WEST'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['NORTH'], ['SOUTH'], ['SOUTH'], ['WATER'], ['WATER'], ['PLANT', 'WHEAT'], ['SOUTH'], ['PLANT', 'WHEAT'], ['SOUTH']], 'market': [['SELL', 'FERTILIZER', 1]]}, {'farmer': ['CARE'], 'hands': [['SOUTH'], ['WATER'], ['DIG'], ['NORTH'], ['NORTH'], ['HARVEST'], ['HARVEST'], ['WEST'], ['WEST'], ['WATER'], ['SOUTH'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['FEED', 'WHEAT'], ['EAST'], ['PLANT', 'WHEAT'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['DIG'], ['DIG'], ['SOUTH'], ['WEST'], ['SOUTH'], ['HARVEST'], ['EAST'], ['EAST']], 'market': [['SELL', 'FERTILIZER', 1]]}, {'farmer': ['WEST'], 'hands': [['CARE'], ['EAST'], ['WATER'], ['CARE'], ['CARE'], ['PLANT', 'WHEAT'], ['PLANT', 'WHEAT'], ['WATER'], ['WEST'], ['HARVEST'], ['DIG'], ['NORTH'], ['EAST']], 'market': [['SELL', 'FERTILIZER', 2]]}, {'farmer': ['WEST'], 'hands': [['COLLECT_FERTILIZER'], ['SOUTH'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['WATER'], ['WATER'], ['WEST'], ['NORTH'], ['DIG'], ['PLANT', 'WHEAT'], ['NORTH'], ['DIG']], 'market': [['SELL', 'FERTILIZER', 1]]}, {'farmer': ['SOUTH'], 'hands': [['WEST'], ['WATER'], ['HARVEST'], ['EAST'], ['EAST'], ['SOUTH'], ['SOUTH'], ['WEST'], ['NORTH'], ['PLANT', 'WHEAT'], ['WATER'], ['WATER'], ['PLANT', 'WHEAT']], 'market': [['SELL', 'FERTILIZER', 1]]}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['NORTH'], ['WEST'], ['DIG'], ['NORTH'], ['SOUTH'], ['HARVEST'], ['HARVEST'], ['WATER'], ['NORTH'], ['WATER'], ['SOUTH'], ['EAST'], ['WATER']], 'market': []}, {'farmer': ['CARE'], 'hands': [['NORTH'], ['WEST'], ['PLANT', 'WHEAT'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['DIG'], ['DIG'], ['WEST'], ['NORTH'], ['EAST'], ['HARVEST'], ['WATER'], ['WEST']], 'market': [['SELL', 'FERTILIZER', 1], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['NORTH'], ['WEST'], ['WATER'], ['CARE'], ['CARE'], ['PLANT', 'WHEAT'], ['PASS'], ['WATER'], ['WATER'], ['SOUTH'], ['DIG'], ['WEST'], ['SOUTH']], 'market': [['SELL', 'FERTILIZER', 2], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['SOUTH'], ['EAST'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['WATER'], ['WATER'], ['NORTH'], ['HARVEST'], ['HARVEST'], ['PASS'], ['NORTH'], ['CARE']], 'market': [['SELL', 'FERTILIZER', 1]]}, {'farmer': ['NORTH'], 'hands': [['WATER'], ['SOUTH'], ['SOUTH'], ['WEST'], ['EAST'], ['PASS'], ['EAST'], ['WATER'], ['PLANT', 'WHEAT'], ['DIG'], ['WATER'], ['DIG'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['EAST'], ['SOUTH'], ['HARVEST'], ['NORTH'], ['FEED', 'WHEAT'], ['PASS'], ['SOUTH'], ['SOUTH'], ['WATER'], ['PASS'], ['EAST'], ['PLANT', 'WHEAT'], ['NORTH']], 'market': [['SELL', 'FERTILIZER', 1]]}, {'farmer': ['WATER'], 'hands': [['EAST'], ['HARVEST'], ['DIG'], ['NORTH'], ['NORTH'], ['PASS'], ['HARVEST'], ['SOUTH'], ['EAST'], ['WATER'], ['SOUTH'], ['WATER'], ['NORTH']], 'market': []}, {'farmer': ['PASS'], 'hands': [['WATER'], ['DIG'], ['PASS'], ['FEED', 'WHEAT'], ['WATER'], ['PASS'], ['DIG'], ['WATER'], ['EAST'], ['PASS'], ['HARVEST'], ['PASS'], ['WATER']], 'market': [['SELL', 'FERTILIZER', 1]]}, {'farmer': ['PASS'], 'hands': [['EAST'], ['PASS'], ['WATER'], ['CARE'], ['EAST'], ['PASS'], ['PASS'], ['PASS'], ['EAST'], ['PASS'], ['DIG'], ['PASS'], ['EAST']], 'market': [['SELL', 'FERTILIZER', 2]]}, {'farmer': ['PASS'], 'hands': [['WATER'], ['WATER'], ['PASS'], ['COLLECT_FERTILIZER'], ['NORTH'], ['PASS'], ['WATER'], ['PASS'], ['WATER'], ['PASS'], ['PASS'], ['PASS'], ['EAST']], 'market': [['SELL', 'FERTILIZER', 1]]}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['PASS'], ['PASS'], ['PASS'], ['WATER'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['WATER'], ['PASS'], ['WATER']], 'market': []}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'STRAWBERRY', 32], ['SELL', 'MILK', 6], ['SELL', 'FERTILIZER', 13], ['SELL', 'WHEAT', 3], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 1], 'hands': [['PICKUP', 'WHEAT', 3], ['WEST'], ['EAST'], ['PICKUP', 'WHEAT', 3], ['PICKUP', 'WHEAT', 3], ['WEST'], ['EAST'], ['PICKUP', 'WHEAT', 3], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['FEED', 'WHEAT'], ['WEST'], ['NORTH'], ['HARVEST'], ['NORTH'], ['WEST'], ['EAST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['FEED', 'WHEAT'], 'hands': [['COLLECT_FERTILIZER'], ['SOUTH'], ['NORTH'], ['FEED', 'WHEAT'], ['HARVEST'], ['NORTH'], ['NORTH'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['EAST'], ['WATER'], ['NORTH'], ['COLLECT_FERTILIZER'], ['FEED', 'WHEAT'], ['NORTH'], ['NORTH'], ['HARVEST'], ['WATER'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['EAST'], 'hands': [['NORTH'], ['WEST'], ['NORTH'], ['NORTH'], ['CARE'], ['WATER'], ['NORTH'], ['FEED', 'WHEAT'], ['HARVEST'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['NORTH'], 'hands': [['FEED', 'WHEAT'], ['WATER'], ['WATER'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['WEST'], ['WATER'], ['COLLECT_FERTILIZER'], ['NORTH'], ['HARVEST']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['EAST'], ['FEED', 'WHEAT'], ['NORTH'], ['WATER'], ['HARVEST'], ['WEST'], ['WATER'], ['WEST']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['EAST'], ['WATER'], ['EAST'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['WEST'], ['NORTH'], ['SOUTH'], ['HARVEST'], ['WATER']], 'market': []}, {'farmer': ['WATER'], 'hands': [['FEED', 'WHEAT'], ['NORTH'], ['EAST'], ['WEST'], ['FEED', 'WHEAT'], ['WATER'], ['WATER'], ['HARVEST'], ['EAST'], ['HARVEST']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['CARE'], ['WATER'], ['SOUTH'], ['NORTH'], ['CARE'], ['SOUTH'], ['HARVEST'], ['FEED', 'WHEAT'], ['SOUTH'], ['WEST']], 'market': []}, {'farmer': ['WATER'], 'hands': [['COLLECT_FERTILIZER'], ['EAST'], ['WATER'], ['NORTH'], ['COLLECT_FERTILIZER'], ['WATER'], ['NORTH'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['WATER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['EAST'], ['WATER'], ['HARVEST'], ['NORTH'], ['EAST'], ['EAST'], ['WATER'], ['WEST'], ['WATER'], ['HARVEST']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['WATER'], ['EAST'], ['SOUTH'], ['FEED', 'WHEAT'], ['SOUTH'], ['NORTH'], ['HARVEST'], ['SOUTH'], ['HARVEST'], ['WEST']], 'market': []}, {'farmer': ['WATER'], 'hands': [['SOUTH'], ['WATER'], ['WATER'], ['CARE'], ['SOUTH'], ['NORTH'], ['EAST'], ['HARVEST'], ['EAST'], ['WATER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['WATER'], ['EAST'], ['NORTH'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['NORTH'], ['WATER'], ['FEED', 'WHEAT'], ['EAST'], ['HARVEST']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['WEST'], ['WATER'], ['NORTH'], ['WEST'], ['WEST'], ['NORTH'], ['HARVEST'], ['CARE'], ['EAST'], ['NORTH']], 'market': []}, {'farmer': ['WATER'], 'hands': [['WEST'], ['EAST'], ['NORTH'], ['WEST'], ['DROP'], ['WATER'], ['SOUTH'], ['EAST'], ['SOUTH'], ['WATER']], 'market': [['SELL', 'MILK', 9], ['SELL', 'FERTILIZER', 2]]}, {'farmer': ['SOUTH'], 'hands': [['CARE'], ['WATER'], ['WATER'], ['PASS'], ['PICKUP', 'WHEAT', 1], ['SOUTH'], ['WATER'], ['EAST'], ['SOUTH'], ['NORTH']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['HARVEST'], ['PASS'], ['EAST'], ['WATER'], ['HARVEST'], ['DROP'], ['WATER'], ['WATER']], 'market': [['SELL', 'WOOL', 8], ['SELL', 'MILK', 3], ['SELL', 'FERTILIZER', 2]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['SOUTH'], ['PASS'], ['PASS'], ['FEED', 'WHEAT'], ['PASS'], ['EAST'], ['WEST'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['WATER'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['WATER'], ['WEST'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [], 'market': [['SELL', 'WHEAT', 51], ['SELL', 'FERTILIZER', 9], ['SELL', 'WOOL', 8], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['HIRE'], ['HIRE'], ['HIRE'], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['PICKUP', 'WHEAT', 1], ['WATER'], ['WEST'], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 3], ['SOUTH'], ['EAST'], ['WEST'], ['WEST'], ['WEST']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['HARVEST'], ['HARVEST'], ['WEST'], ['WEST'], ['NORTH'], ['WATER'], ['NORTH'], ['WEST'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['WEST'], ['HARVEST'], ['FEED', 'WHEAT'], ['SOUTH'], ['NORTH'], ['WEST'], ['NORTH'], ['SOUTH']], 'market': []}, {'farmer': ['WEST'], 'hands': [['EAST'], ['WATER'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['WATER'], ['NORTH'], ['WATER'], ['NORTH'], ['WATER']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['FEED', 'WHEAT'], ['HARVEST'], ['SOUTH'], ['WEST'], ['NORTH'], ['WEST'], ['NORTH'], ['EAST'], ['WATER'], ['WEST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['WATER'], ['FEED', 'WHEAT'], ['FEED', 'WHEAT'], ['WATER'], ['NORTH'], ['NORTH'], ['HARVEST'], ['WEST']], 'market': []}, {'farmer': ['WATER'], 'hands': [['WEST'], ['WATER'], ['WEST'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['WATER'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['WEST'], 'hands': [['DROP'], ['HARVEST'], ['WATER'], ['EAST'], ['EAST'], ['WATER'], ['EAST'], ['NORTH'], ['WATER'], ['WATER']], 'market': [['SELL', 'MILK', 3], ['SELL', 'FERTILIZER', 2]]}, {'farmer': ['WATER'], 'hands': [['EAST'], ['WEST'], ['WEST'], ['EAST'], ['SOUTH'], ['EAST'], ['EAST'], ['WATER'], ['HARVEST'], ['EAST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['WATER'], ['WATER'], ['NORTH'], ['HARVEST'], ['WATER'], ['SOUTH'], ['WEST'], ['WEST'], ['SOUTH']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['NORTH'], ['HARVEST'], ['SOUTH'], ['NORTH'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['SOUTH'], ['WATER'], ['SOUTH'], ['SOUTH']], 'market': []}, {'farmer': ['WATER'], 'hands': [['NORTH'], ['EAST'], ['WATER'], ['COLLECT_FERTILIZER'], ['EAST'], ['WATER'], ['WATER'], ['HARVEST'], ['WATER'], ['SOUTH']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['WATER'], ['EAST'], ['EAST'], ['WEST'], ['FEED', 'WHEAT'], ['WEST'], ['SOUTH'], ['WEST'], ['HARVEST'], ['WATER']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['EAST'], ['SOUTH'], ['EAST'], ['NORTH'], ['COLLECT_FERTILIZER'], ['WATER'], ['SOUTH'], ['WATER'], ['WEST'], ['SOUTH']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['EAST'], ['WATER'], ['WATER'], ['NORTH'], ['WEST'], ['WEST'], ['WATER'], ['HARVEST'], ['NORTH'], ['WATER']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['SOUTH'], ['HARVEST'], ['SOUTH'], ['FEED', 'WHEAT'], ['WEST'], ['WATER'], ['HARVEST'], ['NORTH'], ['WATER'], ['WEST']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['SOUTH'], ['WEST'], ['WEST'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['PASS'], ['EAST'], ['WATER'], ['WEST'], ['NORTH']], 'market': []}, {'farmer': ['WATER'], 'hands': [['WATER'], ['WEST'], ['WEST'], ['EAST'], ['DROP'], ['PASS'], ['WATER'], ['HARVEST'], ['WEST'], ['NORTH']], 'market': [['SELL', 'MILK', 5], ['SELL', 'FERTILIZER', 4]]}, {'farmer': ['PASS'], 'hands': [['EAST'], ['WATER'], ['PASS'], ['SOUTH'], ['PASS'], ['PASS'], ['HARVEST'], ['EAST'], ['SOUTH'], ['NORTH']], 'market': []}, {'farmer': ['PASS'], 'hands': [['WATER'], ['WEST'], ['PASS'], ['SOUTH'], ['PASS'], ['PASS'], ['NORTH'], ['WATER'], ['SOUTH'], ['WATER']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['SOUTH'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['WATER'], ['PASS']], 'market': []}, {'farmer': ['EAST'], 'hands': [], 'market': [['SELL', 'WHEAT', 61], ['SELL', 'FERTILIZER', 7], ['SELL', 'MILK', 3], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['NORTH'], 'hands': [['EAST'], ['WEST'], ['NORTH'], ['WEST'], ['EAST'], ['SOUTH'], ['WEST']], 'market': [['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['HARVEST'], 'hands': [['EAST'], ['NORTH'], ['NORTH'], ['WEST'], ['HARVEST'], ['WATER'], ['WEST'], ['WEST'], ['WEST'], ['WEST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['NORTH'], ['NORTH'], ['NORTH'], ['HARVEST'], ['EAST'], ['HARVEST'], ['WEST'], ['WEST'], ['SOUTH'], ['WEST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['NORTH'], ['NORTH'], ['HARVEST'], ['NORTH'], ['EAST'], ['SOUTH'], ['SOUTH'], ['WEST'], ['WEST'], ['WEST']], 'market': []}, {'farmer': ['HARVEST'], 'hands': [['NORTH'], ['NORTH'], ['EAST'], ['WATER'], ['NORTH'], ['WATER'], ['WATER'], ['WATER'], ['SOUTH'], ['NORTH']], 'market': []}, {'farmer': ['EAST'], 'hands': [['WATER'], ['NORTH'], ['NORTH'], ['HARVEST'], ['WATER'], ['HARVEST'], ['HARVEST'], ['HARVEST'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['HARVEST'], 'hands': [['HARVEST'], ['HARVEST'], ['NORTH'], ['WEST'], ['EAST'], ['SOUTH'], ['WEST'], ['WEST'], ['HARVEST'], ['HARVEST']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['WEST'], ['WEST'], ['WATER'], ['WEST'], ['WATER'], ['WATER'], ['WATER'], ['WATER'], ['WEST'], ['NORTH']], 'market': []}, {'farmer': ['WATER'], 'hands': [['WEST'], ['WATER'], ['HARVEST'], ['WATER'], ['HARVEST'], ['HARVEST'], ['HARVEST'], ['HARVEST'], ['WATER'], ['NORTH']], 'market': []}, {'farmer': ['HARVEST'], 'hands': [['WEST'], ['HARVEST'], ['WEST'], ['HARVEST'], ['WEST'], ['WEST'], ['WEST'], ['SOUTH'], ['HARVEST'], ['NORTH']], 'market': []}, {'farmer': ['WEST'], 'hands': [['WEST'], ['EAST'], ['SOUTH'], ['EAST'], ['WEST'], ['WATER'], ['WATER'], ['WATER'], ['WEST'], ['WATER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['WATER'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['EAST'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['HARVEST'], ['HARVEST'], ['WATER'], ['HARVEST']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['HARVEST'], ['EAST'], ['COLLECT_FERTILIZER'], ['EAST'], ['WEST'], ['WEST'], ['SOUTH'], ['EAST'], ['HARVEST'], ['EAST']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['EAST'], ['SOUTH'], ['WEST'], ['COLLECT_FERTILIZER'], ['WEST'], ['WATER'], ['WATER'], ['EAST'], ['EAST'], ['EAST']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['SOUTH'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['EAST'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['HARVEST'], ['NORTH'], ['EAST'], ['EAST']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['SOUTH'], ['SOUTH'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['EAST'], ['EAST'], ['COLLECT_FERTILIZER'], ['EAST'], ['SOUTH']], 'market': []}, {'farmer': ['WEST'], 'hands': [['SOUTH'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['EAST'], ['EAST'], ['EAST'], ['NORTH'], ['SOUTH']], 'market': []}, {'farmer': ['DROP'], 'hands': [['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['DROP'], ['DROP'], ['DROP'], ['NORTH'], ['EAST'], ['EAST'], ['NORTH'], ['SOUTH']], 'market': [['SELL', 'MILK', 18], ['SELL', 'FERTILIZER', 9], ['SELL', 'WHEAT', 22]]}, {'farmer': ['PASS'], 'hands': [['EAST'], ['DROP'], ['PASS'], ['PASS'], ['PASS'], ['NORTH'], ['EAST'], ['DROP'], ['DROP'], ['SOUTH']], 'market': [['SELL', 'WHEAT', 24], ['SELL', 'MILK', 6], ['SELL', 'FERTILIZER', 3], ['SELL', 'FERTILIZER', 2]]}, {'farmer': ['PASS'], 'hands': [['DROP'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['NORTH'], ['NORTH'], ['PASS'], ['PASS'], ['DROP']], 'market': [['SELL', 'WHEAT', 15], ['SELL', 'FERTILIZER', 1]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['DROP'], ['NORTH'], ['PASS'], ['PASS'], ['PASS']], 'market': [['SELL', 'WHEAT', 15]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['DROP'], ['PASS'], ['PASS'], ['PASS']], 'market': [['SELL', 'WHEAT', 15]]}, {'farmer': ['PASS'], 'hands': [], 'market': []}]
CONFIG = {
    "feed5": True,
    "weed_repair": True,
    "sell_order": True,
    "split": True,
    "banking": True,
    "terminal": True,
    "liq_ramp": True,
    "liq_start": 700,
    "split_caps": {"WHEAT": 10, "FERTILIZER": 5},
    "split_window": (120, 714),
    "bank_value": 2000,
    "bank_window": (120, 679),
    # premium front-run vs a detected mirror: sell my own tape's upcoming
    # premium batches up to fr_horizon turns early (debt-ledgered), so the
    # contested price curve is hit before the mirror's identical batch
    # town-shop-adaptive livestock: convert scheduled COW purchases to SHEEP
    # when unlocked shops make wool demand dominate milk demand
    "town_adapt": True,
    "ta_start": 144,          # day 6 — first two shop draws are known
    "ta_max_convert": 4,
    # staged release of converted-herd wool the tape never scheduled:
    # (last_step, min_price) gates; sell in batches with a minimum gap
    "wool_gates": ((480, 170), (600, 120), (672, 80), (719, 1)),
    "wool_batch": 16,
    "wool_gap": 6,
    "frontrun": True,
    "fr_horizon": 2,
    "fr_window": (240, 700),
    "fr_checkpoints": (216, 240, 264),
    "fr_max_distance": 8,
    "fr_min_price_ratio": 0.4,
}

# ---- market model (exact port of engine 1.32.6) ----
_MI0 = 10000
_PF = 1
_MP = {
    "WHEAT":      (25, 400, "sqrt", 0.8, "log", 0.2),
    "CARROT":     (35, 450, "log", 0.2, "sqrt", 0.7),
    "TOMATO":     (60, 200, "linear", 0.4, "sqrt", 0.6),
    "STRAWBERRY": (120, 100, "sqrt", 0.7, "linear", 1.6),
    "MELON":      (250, 300, "log", 0.2, "sq", 3.6),
    "EGG":        (50, 332, "linear", 0.4, "log", 0.2),
    "MILK":       (160, 122, "sqrt", 0.6, "linear", 1.6),
    "WOOL":       (200, 105, "log", 0.2, "sq", 3.2),
    "FERTILIZER": (100, 200, "linear", 0.4, "linear", 0.4),
}
_PREMIUM = ("MELON", "STRAWBERRY", "MILK", "WOOL")
_MOVES = {"NORTH", "SOUTH", "EAST", "WEST"}
_TILE_OPS = {"PLANT", "BUILD_PASTURE", "BUILD_COOP", "PLACE"}


def _shape(f, x):
    x = max(0.0, x)
    if f == "linear":
        return x
    if f == "sq":
        return x * x
    if f == "sqrt":
        return math.sqrt(x)
    if f == "log":
        return math.log(1.0 + x)
    return x


def _price(item, inv):
    base, T, bf, bt, af, at = _MP[item]
    if inv < _MI0:
        p = base + bt * base / _shape(bf, T) * _shape(bf, _MI0 - inv)
    else:
        p = base - at * base / _shape(af, T) * _shape(af, inv - _MI0)
    return max(_PF, int(round(p)))


def _impact(item, inv, qty):
    return qty * max(0, _price(item, inv) - _price(item, inv + qty))


# ---- per-episode state ----
class _S:
    step_seen = -1
    delay = {}          # actor index -> lag (turns behind the tape)
    debt = {}           # step -> {item: qty already sold early}
    mirror_checks = []  # signature distances at checkpoints
    mirror = False
    converted = 0       # cows converted to sheep so far
    last_wool_sell = -99


def _reset():
    _S.delay = {}
    _S.debt = {}
    _S.mirror_checks = []
    _S.mirror = False
    _S.converted = 0
    _S.last_wool_sell = -99


_MILK_SHOPS = ("PIZZA_SHOP", "ICE_CREAM_SHOP", "SMOOTHIE_SHOP")


def _wool_edge(shops):
    """Daily shop demand for wool minus milk (single-product shops consume 2x)."""
    wool = 12 * sum(1 for s in shops if s == "YARN_STORE")
    milk = 6 * sum(1 for s in shops if s in _MILK_SHOPS)
    return wool - milk


def _tile_counts(farm):
    counts = {}
    for row in farm["tiles"]:
        for t in row:
            if isinstance(t, dict):
                key = t.get("crop") or t.get("animal") or t.get("kind")
                counts[key] = counts.get(key, 0) + 1
    return counts


def _signature_distance(mine, theirs):
    d = abs(len(mine.get("hands", [])) - len(theirs.get("hands", [])))
    d += 3 * abs(len(mine.get("unlocked_quadrants", [])) - len(theirs.get("unlocked_quadrants", [])))
    cm, ct = _tile_counts(mine), _tile_counts(theirs)
    for k in set(cm) | set(ct):
        d += abs(cm.get(k, 0) - ct.get(k, 0))
    return d


def _tape_at(i):
    if 0 <= i < len(TAPE):
        return TAPE[i]
    return {"farmer": ["PASS"], "hands": [], "market": []}


def _actor_action(step, idx, n_hands):
    """Tape action for actor idx (0=farmer, 1..=hands) honoring its lag."""
    lag = _S.delay.get(idx, 0)
    src = _tape_at(step - lag)
    if lag > 0:
        act = _get_actor(src, idx)
        if act and act[0] == "PASS":
            # absorb the lag: skip this PASS, catch up by one
            _S.delay[idx] = lag - 1
            src = _tape_at(step - lag + 1)
    return _get_actor(src, idx)


def _get_actor(entry, idx):
    if idx == 0:
        return list(entry.get("farmer") or ["PASS"])
    hands = entry.get("hands") or []
    if idx - 1 < len(hands):
        h = hands[idx - 1]
        return list(h) if h else ["PASS"]
    return ["PASS"]


def _shed_tiles(board):
    h = board // 2
    return [(h - 1, h - 1), (h, h - 1), (h - 1, h), (h, h)]


def _toward(pos, targets):
    x, y = pos
    tx, ty = min(targets, key=lambda t: abs(t[0] - x) + abs(t[1] - y))
    if x < tx:
        return ["EAST"]
    if x > tx:
        return ["WEST"]
    if y < ty:
        return ["SOUTH"]
    if y > ty:
        return ["NORTH"]
    return None  # already there


def _carried_value(inv, prices):
    return sum(q * prices.get(it, 0) for it, q in (inv or {}).items() if it in _PREMIUM)


def agent(obs):
    step = obs.get("step")
    if step is None:
        step = obs.get("day", 0) * 24 + obs.get("hour", 0)
    if step == 0 or step < _S.step_seen:
        _reset()
    _S.step_seen = step

    me = obs["farms"][obs["player"]]
    board = len(me["tiles"])
    hour = obs.get("hour", step % 24)
    if CONFIG["weed_repair"] and hour == 0:
        _S.delay = {}

    private = obs.get("private") or {}
    shed = dict(private.get("shed") or {})
    invs = private.get("inventories") or [{}]
    prices = (obs.get("market") or {}).get("prices") or {}
    minv = (obs.get("market") or {}).get("inventory") or {}
    positions = [tuple(me["farmer"])] + [tuple(h) for h in me.get("hands", [])]
    n_actors = len(positions)
    sheds = _shed_tiles(board)

    # ---------- field actions ----------
    field = []
    for idx in range(n_actors):
        act = _actor_action(step, idx, n_actors - 1) or ["PASS"]

        if CONFIG["town_adapt"] and act and len(act) > 1 and act[1] == "COW":
            inv = invs[idx] if idx < len(invs) else {}
            if act[0] == "PICKUP" and shed.get("COW", 0) == 0 and shed.get("SHEEP", 0) > 0:
                act = ["PICKUP", "SHEEP"] + act[2:]
            elif act[0] == "PLACE" and not (inv or {}).get("COW") and (inv or {}).get("SHEEP"):
                act = ["PLACE", "SHEEP"] + act[2:]

        if CONFIG["weed_repair"] and act and act[0] in _TILE_OPS:
            x, y = positions[idx]
            tile = me["tiles"][y][x]
            if isinstance(tile, dict) and tile.get("kind") == "WEED":
                act = ["DIG"]
                _S.delay[idx] = _S.delay.get(idx, 0) + 1

        if CONFIG["banking"] and CONFIG["bank_window"][0] <= step <= CONFIG["bank_window"][1]:
            if act[0] == "PASS" or act[0] in _MOVES:
                inv = invs[idx] if idx < len(invs) else {}
                if _carried_value(inv, prices) >= CONFIG["bank_value"]:
                    pos = positions[idx]
                    dist = min(abs(pos[0] - t[0]) + abs(pos[1] - t[1]) for t in sheds)
                    if dist == 0:
                        act = ["DROP"]
                    elif dist <= 1:
                        act = _toward(pos, sheds) or act

        field.append(act)

    # ---------- market actions ----------
    entry = _tape_at(step)
    market = [list(o) for o in (entry.get("market") or [])]

    # subtract debt from earlier pull-forwards
    owed = _S.debt.pop(step, None)
    if owed:
        newm = []
        for o in market:
            if o and o[0] == "SELL" and o[1] in owed and owed[o[1]] > 0:
                q = int(o[2]) if len(o) > 2 else 1
                take = min(q, owed[o[1]])
                owed[o[1]] -= take
                if q - take > 0:
                    newm.append(["SELL", o[1], q - take])
            else:
                newm.append(o)
        market = newm

    # town adaptation: rewrite scheduled cow purchases to sheep on wool demand
    if CONFIG["town_adapt"] and step >= CONFIG["ta_start"]:
        shops = (obs.get("town") or {}).get("unlocked_shops") or []
        if (
            _wool_edge(shops) > 0
            and _S.converted < CONFIG["ta_max_convert"]
            and me["money"] >= 1100  # sheep costs $100 more than the budgeted cow
        ):
            extra = []
            for o in market:
                if o and o[0] == "BUY_ANIMAL" and o[1] == "COW":
                    orig = int(o[2]) if len(o) > 2 else 1
                    q = min(orig, CONFIG["ta_max_convert"] - _S.converted)
                    if q <= 0:
                        break
                    o[1] = "SHEEP"
                    if len(o) > 2:
                        o[2] = q
                    if orig - q > 0:
                        extra.append(["BUY_ANIMAL", "COW", orig - q])
                    _S.converted += q
            market.extend(extra)

    # staged wool release for the converted herd's surplus
    if CONFIG["town_adapt"] and _S.converted > 0 and step - _S.last_wool_sell >= CONFIG["wool_gap"]:
        have = shed.get("WOOL", 0)
        scheduled = sum(
            int(o[2]) if len(o) > 2 else 1
            for o in market
            if o and o[0] == "SELL" and o[1] == "WOOL"
        )
        surplus = have - scheduled
        if surplus > 0 and len(market) < 10:
            gate = next((g for last, g in CONFIG["wool_gates"] if step <= last), 1)
            if prices.get("WOOL", 0) >= gate:
                market.append(["SELL", "WOOL", min(surplus, CONFIG["wool_batch"])])
                _S.last_wool_sell = step

    # mirror detection at fixed checkpoints
    if CONFIG["frontrun"] and step in CONFIG["fr_checkpoints"]:
        opp = obs["farms"][1 - obs["player"]]
        _S.mirror_checks.append(_signature_distance(me, opp))
        if len(_S.mirror_checks) == len(CONFIG["fr_checkpoints"]):
            _S.mirror = all(d <= CONFIG["fr_max_distance"] for d in _S.mirror_checks)

    # premium front-run vs mirror: hit the curve before the clone's batch
    if CONFIG["frontrun"] and _S.mirror and CONFIG["fr_window"][0] <= step <= CONFIG["fr_window"][1]:
        for h in range(1, CONFIG["fr_horizon"] + 1):
            fut = _tape_at(step + h).get("market") or []
            for o in fut:
                if not (o and o[0] == "SELL" and o[1] in _PREMIUM):
                    continue
                item = o[1]
                q = int(o[2]) if len(o) > 2 else 1
                inv_now = minv.get(item, _MI0)
                if _price(item, inv_now) < CONFIG["fr_min_price_ratio"] * _MP[item][0]:
                    continue
                already = sum(
                    int(m[2]) if len(m) > 2 else 1
                    for m in market
                    if m and m[0] == "SELL" and m[1] == item
                )
                q = min(q, max(0, shed.get(item, 0) - already))
                if q > 0 and len(market) < 10:
                    market.append(["SELL", item, q])
                    _S.debt.setdefault(step + h, {})
                    _S.debt[step + h][item] = _S.debt[step + h].get(item, 0) + q

    # split: pull forward next turn's WHEAT/FERT sells
    if CONFIG["split"] and CONFIG["split_window"][0] <= step <= CONFIG["split_window"][1]:
        nxt = _tape_at(step + 1).get("market") or []
        for o in nxt:
            if o and o[0] == "SELL" and o[1] in CONFIG["split_caps"]:
                cap = CONFIG["split_caps"][o[1]]
                q = min(int(o[2]) if len(o) > 2 else 1, cap, shed.get(o[1], 0))
                if q > 0 and len(market) < 10 and _price(o[1], minv.get(o[1], _MI0)) > 2:
                    already = sum(
                        int(m[2]) if len(m) > 2 else 1
                        for m in market
                        if m and m[0] == "SELL" and m[1] == o[1]
                    )
                    q = min(q, max(0, shed.get(o[1], 0) - already))
                    if q > 0:
                        market.append(["SELL", o[1], q])
                        _S.debt.setdefault(step + 1, {})
                        _S.debt[step + 1][o[1]] = _S.debt[step + 1].get(o[1], 0) + q

    # liquidation ramp: sell what the tape forgot, spread to the end
    if CONFIG["liq_ramp"] and CONFIG["liq_start"] <= step <= 718:
        scheduled = {o[1] for o in market if o and o[0] == "SELL"}
        left = max(1, 718 - step)
        for it, q in sorted(shed.items(), key=lambda kv: -prices.get(kv[0], 0) * kv[1]):
            if it in scheduled or it in ("GOOSE", "COW", "SHEEP") or q <= 0:
                continue
            if len(market) >= 10:
                break
            batch = max(1, q // left)
            market.append(["SELL", it, batch])

    # terminal: last 5 turns — bank everything and dump the shed at 718
    if CONFIG["terminal"] and step >= 714:
        for idx in range(n_actors):
            inv = invs[idx] if idx < len(invs) else {}
            carrying = any(v > 0 for v in (inv or {}).values())
            cur = field[idx][0] if field[idx] else "PASS"
            if not carrying or cur not in _MOVES | {"PASS", "DROP", "PLACE", "WATER", "CARE", "FEED"}:
                continue
            pos = positions[idx]
            if pos in sheds:
                field[idx] = ["DROP"]
            else:
                mv = _toward(pos, sheds)
                field[idx] = mv if mv else ["DROP"]
        if step == 718:
            pending = dict(shed)
            for idx in range(n_actors):
                if positions[idx] in sheds:
                    for it, q in (invs[idx] if idx < len(invs) else {}).items():
                        pending[it] = pending.get(it, 0) + q
            market = [o for o in market if o and o[0] == "SELL"]
            listed = {o[1] for o in market}
            for it, q in sorted(pending.items(), key=lambda kv: -prices.get(kv[0], 0) * kv[1]):
                if q > 0 and it not in listed and it not in ("GOOSE", "COW", "SHEEP") and len(market) < 10:
                    market.append(["SELL", it, q])

    # feed5: keep the step-0 feed purchase in slot 0
    if CONFIG["feed5"] and step == 0:
        feed = [o for o in market if o and o[0] == "BUY_PRODUCT" and o[1] == "WHEAT"]
        rest = [o for o in market if not (o and o[0] == "BUY_PRODUCT" and o[1] == "WHEAT")]
        market = feed + rest

    # sell ordering: premium sells first, by self price impact
    if CONFIG["sell_order"] and step > 0:
        prem, other = [], []
        for o in market:
            if o and o[0] == "SELL" and o[1] in _PREMIUM:
                prem.append(o)
            else:
                other.append(o)
        prem.sort(
            key=lambda o: -_impact(o[1], minv.get(o[1], _MI0), int(o[2]) if len(o) > 2 else 1)
        )
        market = prem + other

    return {
        "farmer": field[0] if field else ["PASS"],
        "hands": field[1:],
        "market": market[:10],
    }


kaggriculture_v1 = agent  # keep `agent` as the LAST bound callable below


def _entry(obs):
    return kaggriculture_v1(obs)


agent = _entry  # file runner picks the last newly-bound callable


CONFIG.update({'frontrun': False})
agent = _entry
